# LeetCode 354 — Russian Doll Envelopes (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxEnvelopes(self, envelopes: List[List[int]]) -> int:

```

Bạn có một chồng **phong bì**, mỗi cái có chiều rộng và chiều cao. Một phong bì nhỏ bỏ được vào trong phong bì lớn nếu nó **hẹp hơn hẳn và thấp hơn hẳn**. Câu hỏi: lồng được **nhiều nhất bao nhiêu phong bì** vào nhau, kiểu búp bê Nga (Matryoshka) — cái này trong cái kia trong cái kia nữa?

Người mới thường nghĩ ngay: "thử mọi cách chọn rồi xem cách nào lồng được" — đúng, nhưng với `10^5` phong bì thì không bao giờ chạy xong. Bài này Hard vì lời giải tốt nhất phải ghép **ba ý tưởng** lại với nhau:

1. **Sắp xếp** để biến bài toán 2 chiều (rộng, cao) thành bài toán trên **một dãy**.
2. Một **mẹo sắp xếp rất khéo** (rộng tăng, nhưng cao **giảm** khi rộng bằng nhau) để loại bỏ hoàn toàn chiều rộng.
3. Bài toán kinh điển **Longest Increasing Subsequence (LIS — dãy con tăng dài nhất, LeetCode 300)**, giải trong `O(n log n)` bằng **patience sorting** + **tìm kiếm nhị phân**.

Notebook dạy cả ba ý **từ con số 0**, mỗi ý có ví dụ nhỏ chạy được trước khi dùng.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề, kiểm tra bằng tay "phong bì A có lồng vào B không", hiểu chữ **"hẳn"** (bằng nhau thì **không** lồng) và luật **không được xoay**.
2. **Cách 1 — Brute force**: (a) thử mọi tập con (làm "trọng tài"), (b) xem phong bì như **đồ thị** và dùng **DFS (đệ quy)** tìm đường dài nhất; thấy vì sao cần **memo** và vì sao vẫn chậm.
3. **Cách 2 — Sắp xếp theo chiều rộng + DP LIS `O(n²)`**: học **dãy con (subsequence)** và **LIS** từ căn bản; thấy rõ **vấn đề chiều rộng bằng nhau**.
4. **Cách 3 — Mẹo sắp xếp `(w tăng, h giảm)`**: giải thích kỹ vì sao `h` giảm trong cùng một `w` loại được các cặp cùng rộng; **demo sai** khi sắp `h` tăng.
5. **Cách 4 (cách chính) — LIS `O(n log n)`** bằng mảng `tails` + `bisect_left`; trace bảng `tails` từng bước; hiểu vì sao **`tails` không phải dãy thật**; khôi phục chuỗi phong bì thật nếu cần.
6. Code nộp LeetCode, test đối chiếu chéo các cách trên input ngẫu nhiên, vì sao đúng, so sánh độ phức tạp, lỗi hay gặp, mẹo nhận dạng pattern, biến thể và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

You are given a 2D array of integers `envelopes` where `envelopes[i] = [wi, hi]` represents the width and the height of an envelope.

One envelope can fit into another if and only if **both the width and height** of one envelope are **greater than** the other envelope's width and height.

Return *the maximum number of envelopes you can Russian doll (i.e., put one inside the other)*.

**Note:** You cannot rotate an envelope.

**Example 1**

```text
Input:  envelopes = [[5,4],[6,4],[6,7],[2,3]]
Output: 3
Explanation: The maximum number of envelopes you can Russian doll is 3 ([2,3] => [5,4] => [6,7]).
```

**Example 2**

```text
Input:  envelopes = [[1,1],[1,1],[1,1]]
Output: 1
```

**Constraints**

- `1 <= envelopes.length <= 10^5`
- `envelopes[i].length == 2`
- `1 <= wi, hi <= 10^5`

**Dịch nhanh:** mỗi phong bì là cặp `[w, h]` (w = width = rộng, h = height = cao). Phong bì `A` lồng được vào `B` **khi và chỉ khi** `A.w < B.w` **và** `A.h < B.h` (nhỏ hơn **hẳn** ở **cả hai** chiều). Không được xoay (không được đổi chỗ `w` và `h`). Trả về số phong bì nhiều nhất trong một chuỗi lồng nhau.

Hai con số cần nhớ từ constraints: `n` tới `10^5` → thuật toán `O(n²)` (khoảng `10^10` phép) chắc chắn quá chậm; ta cần cỡ `O(n log n)`.

## 2. Hiểu đề bằng tay

Trước khi nghĩ thuật toán, ta viết một hàm nhỏ trả lời câu hỏi cơ bản nhất: **"phong bì `a` có lồng được vào phong bì `b` không?"**

```python
def fits(a, b):
    return a[0] < b[0] and a[1] < b[1]
```

- `a[0]` là chiều rộng của `a`, `a[1]` là chiều cao của `a` (list trong Python đánh chỉ số từ 0).
- Dấu `<` (nhỏ hơn hẳn), **không phải** `<=`: hai phong bì rộng bằng nhau thì không cái nào lồng vào cái nào.

Với Example 1 `[[5,4],[6,4],[6,7],[2,3]]`, ta thử mọi cặp `(a, b)`:

| a | b | `a.w < b.w`? | `a.h < b.h`? | Lồng được? |
|---|---|:---:|:---:|:---:|
| `[2,3]` | `[5,4]` | 2<5 ✓ | 3<4 ✓ | **Có** |
| `[2,3]` | `[6,4]` | 2<6 ✓ | 3<4 ✓ | **Có** |
| `[2,3]` | `[6,7]` | 2<6 ✓ | 3<7 ✓ | **Có** |
| `[5,4]` | `[6,7]` | 5<6 ✓ | 4<7 ✓ | **Có** |
| `[5,4]` | `[6,4]` | 5<6 ✓ | 4<4 ✗ | Không (cao bằng nhau) |
| `[6,4]` | `[6,7]` | 6<6 ✗ | 4<7 ✓ | Không (rộng bằng nhau) |

Chuỗi dài nhất: `[2,3] → [5,4] → [6,7]`, gồm **3** phong bì. Cell dưới kiểm chứng bằng code.

In [1]:
def fits(a, b):
    """True nếu phong bì a lồng được vào phong bì b (nhỏ hơn hẳn ở cả 2 chiều)."""
    return a[0] < b[0] and a[1] < b[1]

envelopes = [[5, 4], [6, 4], [6, 7], [2, 3]]
for a in envelopes:
    for b in envelopes:
        if fits(a, b):
            print(f"{a} lồng được vào {b}")

print()
print("fits([5,4],[6,4]) =", fits([5, 4], [6, 4]), " <- cao bằng nhau")
print("fits([6,4],[6,7]) =", fits([6, 4], [6, 7]), " <- rộng bằng nhau")
print("fits([1,1],[1,1]) =", fits([1, 1], [1, 1]), " <- Example 2: giống hệt nhau")

[5, 4] lồng được vào [6, 7]
[2, 3] lồng được vào [5, 4]
[2, 3] lồng được vào [6, 4]
[2, 3] lồng được vào [6, 7]

fits([5,4],[6,4]) = False  <- cao bằng nhau
fits([6,4],[6,7]) = False  <- rộng bằng nhau
fits([1,1],[1,1]) = False  <- Example 2: giống hệt nhau


### Ba quan sát quan trọng trước khi giải

**Quan sát 1 — Thứ tự trong input không quan trọng.** Đề cho `envelopes` theo một thứ tự bất kỳ, nhưng ta được **tự chọn** phong bì nào bỏ vào phong bì nào. Khác với bài "dãy con" thông thường (phải giữ thứ tự gốc), ở đây ta được tự do sắp xếp lại. Đây chính là cửa ngõ cho mẹo **sắp xếp** ở phần sau.

**Quan sát 2 — Một chuỗi hợp lệ chỉ cần kiểm tra các cặp liền kề.** Nếu `A` lồng vào `B` và `B` lồng vào `C` thì chắc chắn `A` lồng vào `C` (vì `A.w < B.w < C.w` và `A.h < B.h < C.h`). Tính chất này gọi là **bắc cầu (transitive)**. Nên để kiểm tra chuỗi `[e0, e1, e2, ...]` ta chỉ cần kiểm `fits(e0, e1)`, `fits(e1, e2)`, ...

**Quan sát 3 — Đáp án luôn ≥ 1** (vì `n ≥ 1`, một phong bì đứng một mình là chuỗi dài 1). Example 2 cho thấy các phong bì **giống hệt nhau** không lồng được vào nhau, nên đáp án là 1.

Cell dưới viết hàm `is_valid_chain` dùng quan sát 2 — ta sẽ dùng nó để kiểm chứng các đáp án sau này.

In [2]:
def is_valid_chain(chain):
    """Chuỗi phong bì (theo thứ tự trong -> ngoài) có hợp lệ không? Chỉ cần kiểm cặp liền kề."""
    return all(fits(chain[i], chain[i + 1]) for i in range(len(chain) - 1))

print(is_valid_chain([[2, 3], [5, 4], [6, 7]]))   # True  — chuỗi của Example 1
print(is_valid_chain([[2, 3], [5, 4], [6, 4]]))   # False — [5,4] -> [6,4] cao bằng nhau
print(is_valid_chain([[1, 1]]))                   # True  — 1 phong bì luôn hợp lệ
print(is_valid_chain([[3, 2], [2, 3]]))           # False — không được xoay [3,2] thành [2,3]

True
False
True
False


## 3. Cách 1a — Thử mọi tập con (trọng tài)

Cách người mới nghĩ ra đầu tiên: **thử mọi cách chọn một nhóm phong bì**, xem nhóm nào lồng được thành một chuỗi, lấy nhóm lớn nhất.

Nhưng một nhóm phong bì thì xếp theo thứ tự nào để kiểm tra? Nếu nhóm lồng được thì phong bì trong cùng có chiều rộng nhỏ nhất, kế tiếp lớn hơn,... nên ta chỉ cần **sắp nhóm theo chiều rộng** rồi gọi `is_valid_chain`.

### Kiến thức nền: `itertools.combinations` — liệt kê mọi nhóm `r` phần tử

`combinations(items, r)` sinh ra **mọi cách chọn `r` phần tử** (không lặp, không quan tâm thứ tự) từ `items`. Với `n` phần tử, tổng số nhóm (mọi `r` từ 1 đến n) là `2^n - 1`.

In [3]:
from itertools import combinations

items = ["A", "B", "C"]
for r in range(1, len(items) + 1):
    print(r, "->", list(combinations(items, r)))

total = sum(1 for r in range(1, 4) for _ in combinations(items, r))
print("Tổng số nhóm khác rỗng:", total, "= 2^3 - 1 =", 2**3 - 1)

1 -> [('A',), ('B',), ('C',)]
2 -> [('A', 'B'), ('A', 'C'), ('B', 'C')]
3 -> [('A', 'B', 'C')]
Tổng số nhóm khác rỗng: 7 = 2^3 - 1 = 7


Bây giờ ghép lại thành hàm brute force. Để không phụ thuộc thứ tự đầu vào, ta sắp `sorted(envelopes)` một lần (Python sắp list các cặp theo `w` trước, `w` bằng nhau thì theo `h`); mọi nhóm do `combinations` sinh ra sẽ giữ đúng thứ tự đã sắp đó.

Hàm này **cực chậm** nhưng **cực dễ tin là đúng**, nên ta gọi nó là **trọng tài**: mọi cách nhanh hơn phía sau đều được so với nó trên input nhỏ.

In [4]:
def max_envelopes_subsets(envelopes):
    """Trọng tài: thử mọi tập con, O(2^n · n). Chỉ dùng cho n nhỏ (≤ ~15)."""
    envs = sorted(envelopes)            # theo w tăng, w bằng thì theo h tăng
    best = 0
    for r in range(1, len(envs) + 1):
        for group in combinations(envs, r):   # group giữ thứ tự đã sắp
            if is_valid_chain(group):
                best = max(best, r)
    return best

print(max_envelopes_subsets([[5, 4], [6, 4], [6, 7], [2, 3]]))   # 3
print(max_envelopes_subsets([[1, 1], [1, 1], [1, 1]]))           # 1
print(max_envelopes_subsets([[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]]))  # 4

3
1
4


**Cách 1a vướng ở đâu?** Số tập con là `2^n`. Với `n = 30` đã hơn một tỷ nhóm; với `n = 10^5` thì con số này lớn hơn số nguyên tử trong vũ trụ. Ta cần **tận dụng cấu trúc** của bài thay vì thử mù.

Cấu trúc đầu tiên: quan hệ "lồng vào" giữa các cặp phong bì tạo thành một **đồ thị**.

## 4. Kiến thức nền: đồ thị (graph) và danh sách kề (adjacency list)

**Đồ thị** gồm các **đỉnh (vertex/node)** và các **cạnh (edge)** nối chúng. Ở đây:

- Mỗi phong bì là **một đỉnh** (đánh số theo chỉ số `i` trong list).
- Có **cạnh có hướng** `i → j` nếu phong bì `i` lồng được vào phong bì `j`.

Một chuỗi lồng nhau `[2,3] → [5,4] → [6,7]` chính là **một đường đi** theo các mũi tên. Bài toán trở thành: **tìm đường đi có nhiều đỉnh nhất**.

Cách lưu đồ thị phổ biến nhất là **danh sách kề**: `graph[i]` là list các đỉnh `j` mà từ `i` đi tới được. Với Example 1 (chỉ số: `0=[5,4]`, `1=[6,4]`, `2=[6,7]`, `3=[2,3]`):

```text
graph[0] = [2]          # [5,4] -> [6,7]
graph[1] = []           # [6,4] không lồng vào ai
graph[2] = []           # [6,7] là to nhất
graph[3] = [0, 1, 2]    # [2,3] lồng vào cả 3 cái còn lại
```

Đồ thị này **không có chu trình** (không thể đi vòng về chỗ cũ), vì mỗi bước đi `w` tăng hẳn — không thể tăng mãi rồi quay lại số cũ. Đồ thị có hướng không chu trình gọi là **DAG (Directed Acyclic Graph)**. Nhờ không có chu trình, đệ quy trên nó luôn dừng.

In [5]:
def build_graph(envelopes):
    n = len(envelopes)
    graph = [[] for _ in range(n)]      # n list rỗng, KHÔNG dùng [[]] * n (sẽ dùng chung 1 list!)
    for i in range(n):
        for j in range(n):
            if fits(envelopes[i], envelopes[j]):
                graph[i].append(j)
    return graph

envelopes = [[5, 4], [6, 4], [6, 7], [2, 3]]
graph = build_graph(envelopes)
for i, nxt in enumerate(graph):
    print(f"graph[{i}] {envelopes[i]} -> {[envelopes[j] for j in nxt]}")

graph[0] [5, 4] -> [[6, 7]]
graph[1] [6, 4] -> []
graph[2] [6, 7] -> []
graph[3] [2, 3] -> [[5, 4], [6, 4], [6, 7]]


## 5. Kiến thức nền: đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn. Mỗi hàm đệ quy cần 2 phần:

1. **Trường hợp cơ sở (base case)**: bài đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: giải bài lớn bằng cách gọi bài nhỏ hơn rồi ghép kết quả.

Ví dụ mini: một "chuỗi" `next_of[i]` cho biết sau `i` là ai (`None` = hết). Độ dài chuỗi bắt đầu từ `i`:

```text
length(i) = 1                        nếu next_of[i] là None   (base case)
length(i) = 1 + length(next_of[i])   ngược lại                (bước đệ quy)
```

Với bài phong bì, ta làm giống hệt nhưng mỗi đỉnh có **nhiều** "người kế tiếp", nên lấy **max**:

```text
longest_from(i) = 1 + max( longest_from(j) for j in graph[i] )   # max của rỗng = 0
```

`longest_from(i)` = số phong bì nhiều nhất của một chuỗi **bắt đầu (trong cùng) bằng phong bì `i`**.

In [6]:
next_of = {0: 1, 1: 2, 2: 3, 3: None}     # 0 -> 1 -> 2 -> 3

def length(i):
    if next_of[i] is None:                # base case
        return 1
    return 1 + length(next_of[i])         # gọi lại chính nó với đỉnh kế tiếp

for start in range(4):
    print(f"length({start}) = {length(start)}")

length(0) = 4
length(1) = 3
length(2) = 2
length(3) = 1


## 6. Cách 1b — DFS tìm đường dài nhất trên đồ thị

**DFS (Depth-First Search — tìm kiếm theo chiều sâu)**: từ một đỉnh, đi sâu hết mức theo từng mũi tên rồi mới quay lại thử mũi tên khác. Đệ quy `longest_from` ở trên chính là DFS.

Đáp án của bài = `max(longest_from(i) for i in range(n))` — vì chuỗi tốt nhất phải bắt đầu ở **một** phong bì nào đó.

Tính tay Example 1:

- `longest_from(2)` (`[6,7]`): không đi đâu được → `1`.
- `longest_from(1)` (`[6,4]`): không đi đâu được → `1`.
- `longest_from(0)` (`[5,4]`): đi tới `2` → `1 + 1 = 2`.
- `longest_from(3)` (`[2,3]`): đi tới `0, 1, 2` → `1 + max(2, 1, 1) = 3`.

Đáp án `max(2, 1, 1, 3) = 3`. ✓

In [7]:
def max_envelopes_dfs(envelopes):
    """DFS thuần, KHÔNG nhớ kết quả: đúng nhưng có thể tốn thời gian mũ."""
    graph = build_graph(envelopes)

    def longest_from(i):
        best_next = 0
        for j in graph[i]:
            best_next = max(best_next, longest_from(j))
        return 1 + best_next

    return max(longest_from(i) for i in range(len(envelopes)))

print(max_envelopes_dfs([[5, 4], [6, 4], [6, 7], [2, 3]]))   # 3
print(max_envelopes_dfs([[1, 1], [1, 1], [1, 1]]))           # 1

3
1


### Trace DFS: nhìn thấy các lời gọi bị lặp

Hàm trace dưới in mỗi lời gọi `longest_from(i)` với độ thụt lề bằng độ sâu đệ quy, và in kết quả khi trả về. Để ý: `longest_from(2)` (phong bì `[6,7]`) được tính **nhiều lần** — một lần khi bắt đầu từ `0`, một lần trong nhánh `3 → 0 → 2`, một lần trong nhánh `3 → 2`, và một lần khi bắt đầu từ chính `2`.

In [8]:
def trace_dfs(envelopes):
    graph = build_graph(envelopes)
    calls = {"count": 0}

    def longest_from(i, depth):
        calls["count"] += 1
        pad = "    " * depth
        print(f"{pad}longest_from({i}) {envelopes[i]}  -> đi tới {graph[i]}")
        best_next = 0
        for j in graph[i]:
            best_next = max(best_next, longest_from(j, depth + 1))
        print(f"{pad}= {1 + best_next}")
        return 1 + best_next

    ans = 0
    for i in range(len(envelopes)):
        ans = max(ans, longest_from(i, 0))
    print("Đáp án:", ans, "| tổng số lời gọi:", calls["count"])

trace_dfs([[5, 4], [6, 4], [6, 7], [2, 3]])

longest_from(0) [5, 4]  -> đi tới [2]
    longest_from(2) [6, 7]  -> đi tới []
    = 1
= 2
longest_from(1) [6, 4]  -> đi tới []
= 1
longest_from(2) [6, 7]  -> đi tới []
= 1
longest_from(3) [2, 3]  -> đi tới [0, 1, 2]
    longest_from(0) [5, 4]  -> đi tới [2]
        longest_from(2) [6, 7]  -> đi tới []
        = 1
    = 2
    longest_from(1) [6, 4]  -> đi tới []
    = 1
    longest_from(2) [6, 7]  -> đi tới []
    = 1
= 3
Đáp án: 3 | tổng số lời gọi: 9


### Cách 1b vướng ở đâu? Số lời gọi bùng nổ

Với 4 phong bì thì lặp vài lần chẳng sao. Nhưng thử trường hợp xấu nhất: `n` phong bì **lồng được hết vào nhau**, `[[1,1],[2,2],...,[n,n]]`. Khi đó đỉnh `i` đi tới **mọi** đỉnh lớn hơn, và mỗi đỉnh lại tự gọi toàn bộ phần sau. Gọi `C(k)` là số lời gọi khi đứng ở đỉnh có `k` đỉnh lớn hơn: `C(k) = 1 + C(0) + C(1) + ... + C(k-1) = 2^k`. Cộng cho mọi điểm xuất phát được `2^n - 1` lời gọi — **tăng theo hàm mũ**, y hệt brute force tập con.

In [9]:
def count_dfs_calls(envelopes):
    graph = build_graph(envelopes)
    calls = 0

    def longest_from(i):
        nonlocal calls          # nonlocal: cho phép hàm con sửa biến 'calls' của hàm ngoài
        calls += 1
        return 1 + max((longest_from(j) for j in graph[i]), default=0)

    for i in range(len(envelopes)):
        longest_from(i)
    return calls

for n in [4, 8, 12, 16]:
    chain = [[k, k] for k in range(1, n + 1)]
    print(f"n = {n:2d}: số lời gọi = {count_dfs_calls(chain):6d}  (2^n - 1 = {2**n - 1})")

n =  4: số lời gọi =     15  (2^n - 1 = 15)
n =  8: số lời gọi =    255  (2^n - 1 = 255)
n = 12: số lời gọi =   4095  (2^n - 1 = 4095)
n = 16: số lời gọi =  65535  (2^n - 1 = 65535)


### Sửa nhanh: memo (ghi nhớ kết quả)

Kết quả của `longest_from(i)` **chỉ phụ thuộc vào `i`** — gọi bao nhiêu lần cũng ra cùng một số. Vậy lần đầu tính xong thì **ghi vào sổ** (`memo[i]`), lần sau chỉ việc đọc ra. Kỹ thuật này gọi là **memoization** (ghi nhớ) — viên gạch đầu tiên của **quy hoạch động (Dynamic Programming, DP)**.

```python
memo = {}
def longest_from(i):
    if i in memo:            # đã tính rồi -> trả ngay
        return memo[i]
    ... tính như cũ ...
    memo[i] = result         # ghi sổ trước khi trả
    return result
```

(Python còn có sẵn `@functools.lru_cache` làm việc này tự động; ở đây viết tay bằng `dict` để thấy rõ.)

Với memo, mỗi đỉnh chỉ thực sự "làm việc" một lần, mỗi cạnh chỉ được duyệt một lần. Nhưng việc **xây đồ thị** đã phải thử mọi cặp `(i, j)` → `O(n²)` thời gian, và đồ thị có thể có tới `~n²/2` cạnh → `O(n²)` bộ nhớ.

In [10]:
def max_envelopes_dfs_memo(envelopes):
    graph = build_graph(envelopes)          # O(n²)
    memo = {}

    def longest_from(i):
        if i in memo:
            return memo[i]
        best_next = 0
        for j in graph[i]:
            best_next = max(best_next, longest_from(j))
        memo[i] = 1 + best_next
        return memo[i]

    return max(longest_from(i) for i in range(len(envelopes)))

print(max_envelopes_dfs_memo([[5, 4], [6, 4], [6, 7], [2, 3]]))   # 3
chain16 = [[k, k] for k in range(1, 17)]
print(max_envelopes_dfs_memo(chain16), "<- 16 phong bì lồng hết, giờ chạy tức thì")

3
16 <- 16 phong bì lồng hết, giờ chạy tức thì


**Vẫn vướng:** `O(n²)` với `n = 10^5` là `10^10` cặp — không thể. Và đồ thị `~5·10^9` cạnh thì cũng không có đủ bộ nhớ. Nút thắt là ta đang **so mọi cặp** một cách mù quáng. Bước tiếp theo: **sắp xếp** để biết trước "ai có thể đứng trước ai".

## 7. Bước đột phá 1 — Sắp xếp theo chiều rộng

**Ý chính:** trong mọi chuỗi lồng nhau, chiều rộng **tăng dần hẳn** từ trong ra ngoài. Vậy nếu ta **sắp toàn bộ phong bì theo `w` tăng dần**, thì mọi chuỗi hợp lệ đều xuất hiện **theo đúng thứ tự trái → phải** trong list đã sắp. Nói cách khác, sau khi sắp, ta **không cần quay lại nhìn bên trái** nữa: phong bì đứng sau trong chuỗi luôn nằm ở vị trí lớn hơn trong list.

Ví dụ Example 1 sau khi sắp theo `w`: `[2,3], [5,4], [6,4], [6,7]` (hai cái `w = 6` thứ tự nào cũng được — đây sẽ là rắc rối ở phần 9). Chuỗi `[2,3] → [5,4] → [6,7]` là các vị trí `0, 1, 3` — tăng dần.

### Kiến thức nền: `sorted` với `key=lambda`

- `sorted(lst)` trả về list mới đã sắp, không đổi `lst`; `lst.sort()` sắp **tại chỗ**.
- `key=` là một hàm "chấm điểm" mỗi phần tử; Python sắp theo điểm đó.
- `lambda e: e[0]` là cách viết nhanh của hàm `def f(e): return e[0]` — chấm điểm bằng chiều rộng.
- Khi hai phần tử có **cùng điểm**, `sorted` giữ nguyên thứ tự ban đầu của chúng (gọi là **sắp xếp ổn định — stable sort**).

In [11]:
envelopes = [[5, 4], [6, 4], [6, 7], [2, 3]]

print("theo w        :", sorted(envelopes, key=lambda e: e[0]))
print("mặc định      :", sorted(envelopes))          # so cả cặp: w trước, rồi tới h
print("list gốc      :", envelopes)                  # sorted() không đổi list gốc

# Stable sort: [6,7] đứng trước [6,4] trong input -> vẫn đứng trước sau khi sắp theo w
print("stable        :", sorted([[6, 7], [2, 3], [6, 4]], key=lambda e: e[0]))

theo w        : [[2, 3], [5, 4], [6, 4], [6, 7]]
mặc định      : [[2, 3], [5, 4], [6, 4], [6, 7]]
list gốc      : [[5, 4], [6, 4], [6, 7], [2, 3]]
stable        : [[2, 3], [6, 7], [6, 4]]


## 8. Kiến thức nền: dãy con (subsequence) và bài LIS (LeetCode 300)

Sau khi sắp, bài phong bì trở thành: **chọn ra một dãy con** của list sao cho mỗi phần tử "lớn hơn hẳn" phần tử trước. Đây đúng là dạng của bài kinh điển **LIS**, nên ta học LIS cho thật chắc trên một dãy số đơn giản.

**Dãy con (subsequence)**: lấy ra **một số phần tử** của dãy, **giữ nguyên thứ tự**, được phép **bỏ qua** phần tử ở giữa. Khác với **đoạn con (subarray)** phải **liền nhau**.

- Dãy `[10, 9, 2, 5, 3, 7, 101, 18]`
- `[2, 5, 7, 101]` là dãy con (bỏ qua 10, 9, 3, 18) ✓ — và nó **tăng hẳn**.
- `[5, 2]` **không** phải dãy con (đảo thứ tự).
- `[2, 5, 3]` là đoạn con (vị trí 2, 3, 4 liền nhau) và cũng là dãy con.

**LIS — Longest (Strictly) Increasing Subsequence**: dãy con **tăng hẳn** dài nhất. Với `[10, 9, 2, 5, 3, 7, 101, 18]`, đáp án là **4** (ví dụ `[2, 3, 7, 18]` hoặc `[2, 5, 7, 101]`). Đây chính là Example 1 của LeetCode 300.

In [12]:
nums = [10, 9, 2, 5, 3, 7, 101, 18]

def is_strictly_increasing(seq):
    return all(seq[i] < seq[i + 1] for i in range(len(seq) - 1))

def lis_bruteforce(nums):
    """Trọng tài cho LIS: thử mọi dãy con (combinations giữ nguyên thứ tự vị trí)."""
    best = 0
    for r in range(1, len(nums) + 1):
        for sub in combinations(nums, r):
            if is_strictly_increasing(sub):
                best = max(best, r)
    return best

print(is_strictly_increasing([2, 5, 7, 101]))   # True
print(is_strictly_increasing([2, 5, 5, 7]))     # False — 5, 5 không tăng HẲN
print("LIS =", lis_bruteforce(nums))             # 4

True
False
LIS = 4


### DP cho LIS trong `O(n²)`

Định nghĩa: **`dp[i]` = độ dài dãy con tăng dài nhất KẾT THÚC tại vị trí `i`** (bắt buộc lấy `nums[i]` làm phần tử cuối).

Công thức: dãy kết thúc ở `i` có phần tử ngay trước là một `nums[j]` nào đó với `j < i` và `nums[j] < nums[i]`, hoặc chỉ gồm một mình `nums[i]`:

```text
dp[i] = 1 + max( dp[j]  for j < i if nums[j] < nums[i] )     # max của rỗng = 0
đáp án = max(dp)
```

Tính tay với `nums = [10, 9, 2, 5, 3, 7, 101, 18]`:

| i | nums[i] | các j hợp lệ (nums[j] < nums[i]) | dp[i] |
|---:|---:|---|---:|
| 0 | 10 | — | 1 |
| 1 | 9 | — | 1 |
| 2 | 2 | — | 1 |
| 3 | 5 | j=2 (2) | 2 |
| 4 | 3 | j=2 (2) | 2 |
| 5 | 7 | j=2,3,4 (dp 1,2,2) | 3 |
| 6 | 101 | j=0..5 (dp lớn nhất 3) | 4 |
| 7 | 18 | j=0..5 (dp lớn nhất 3) | 4 |

`max(dp) = 4`. Đây cũng chính là `longest_from` của phần 6, chỉ là đọc theo chiều ngược lại ("kết thúc tại" thay vì "bắt đầu từ") và **không cần xây đồ thị**, vì sau khi sắp ta biết mọi "mũi tên" đều đi từ trái sang phải.

In [13]:
def lis_dp(nums, verbose=False):
    n = len(nums)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
        if verbose:
            print(f"i={i}, nums[i]={nums[i]:3d} -> dp[i]={dp[i]}   dp={dp[:i + 1]}")
    return max(dp) if dp else 0

print("LIS =", lis_dp([10, 9, 2, 5, 3, 7, 101, 18], verbose=True))

i=0, nums[i]= 10 -> dp[i]=1   dp=[1]
i=1, nums[i]=  9 -> dp[i]=1   dp=[1, 1]
i=2, nums[i]=  2 -> dp[i]=1   dp=[1, 1, 1]
i=3, nums[i]=  5 -> dp[i]=2   dp=[1, 1, 1, 2]
i=4, nums[i]=  3 -> dp[i]=2   dp=[1, 1, 1, 2, 2]
i=5, nums[i]=  7 -> dp[i]=3   dp=[1, 1, 1, 2, 2, 3]
i=6, nums[i]=101 -> dp[i]=4   dp=[1, 1, 1, 2, 2, 3, 4]
i=7, nums[i]= 18 -> dp[i]=4   dp=[1, 1, 1, 2, 2, 3, 4, 4]
LIS = 4


## 9. Cách 2 — Sắp theo `w` + DP LIS `O(n²)` (so cả hai chiều)

Ghép phần 7 và 8: sắp phong bì theo `w` tăng, rồi chạy DP LIS, nhưng "lớn hơn hẳn" bây giờ nghĩa là **`fits(envs[j], envs[i])`** — so **cả `w` lẫn `h`**:

```text
dp[i] = 1 + max( dp[j]  for j < i if envs[j].w < envs[i].w and envs[j].h < envs[i].h )
```

**Tại sao vẫn phải so `w`, đã sắp rồi mà?** Vì sắp theo `w` chỉ đảm bảo `w` **không giảm** (`envs[j].w <= envs[i].w`), chưa đảm bảo **tăng hẳn**. Các phong bì **cùng rộng** đứng cạnh nhau, và chúng **không được** lồng vào nhau. Điều kiện `envs[j].w < envs[i].w` chặn chúng lại.

### Trace Cách 2 trên Example 1

Sau khi sắp theo `w` (stable, giữ thứ tự input cho hai cái `w = 6`): `[2,3], [5,4], [6,4], [6,7]`. Hàm trace in cho mỗi `i` những `j` nào "lồng được vào `i`" và giá trị `dp[i]`. Để ý ở `i = 3` (`[6,7]`): `j = 2` (`[6,4]`) có `h` nhỏ hơn nhưng **bị loại vì cùng `w = 6`**.

In [14]:
def max_envelopes_dp(envelopes):
    """Sắp theo w, DP LIS O(n²) kiểm tra cả hai chiều."""
    envs = sorted(envelopes, key=lambda e: e[0])
    n = len(envs)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if envs[j][0] < envs[i][0] and envs[j][1] < envs[i][1]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

print(max_envelopes_dp([[5, 4], [6, 4], [6, 7], [2, 3]]))                  # 3
print(max_envelopes_dp([[1, 1], [1, 1], [1, 1]]))                          # 1
print(max_envelopes_dp([[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]]))          # 4


def trace_dp_envelopes(envelopes):
    envs = sorted(envelopes, key=lambda e: e[0])
    print("Sau khi sắp theo w:", envs)
    dp = [1] * len(envs)
    for i in range(len(envs)):
        ok, rejected = [], []
        for j in range(i):
            if envs[j][0] < envs[i][0] and envs[j][1] < envs[i][1]:
                ok.append(j)
                dp[i] = max(dp[i], dp[j] + 1)
            else:
                rejected.append(j)
        print(f"i={i} {envs[i]}: j lồng được = {ok}, j bị loại = {rejected} -> dp[{i}] = {dp[i]}")
    print("dp =", dp, "-> đáp án", max(dp))

trace_dp_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]])

3
1
4
Sau khi sắp theo w: [[2, 3], [5, 4], [6, 4], [6, 7]]
i=0 [2, 3]: j lồng được = [], j bị loại = [] -> dp[0] = 1
i=1 [5, 4]: j lồng được = [0], j bị loại = [] -> dp[1] = 2
i=2 [6, 4]: j lồng được = [0], j bị loại = [1] -> dp[2] = 2
i=3 [6, 7]: j lồng được = [0, 1], j bị loại = [2] -> dp[3] = 3
dp = [1, 2, 2, 3] -> đáp án 3


### Vấn đề chiều rộng bằng nhau: nếu chỉ so `h` thì sai

Cách 2 phải so **hai** chiều. Ước mơ tự nhiên: "đã sắp theo `w` rồi, giá mà chỉ cần chạy **LIS một chiều trên dãy `h`** thì sẽ dùng được các thuật toán LIS nhanh". Thử ngây thơ: sắp theo `(w tăng, h tăng)` rồi lấy LIS của dãy `h`.

Phản ví dụ: `[[1,1],[1,2],[1,3]]` — ba phong bì **cùng rộng 1**, không cái nào lồng vào cái nào, đáp án đúng là **1**. Nhưng dãy `h` sau khi sắp là `[1, 2, 3]`, tăng hẳn, LIS = **3**. Sai! LIS đã "chọn" cả ba phong bì cùng rộng, vì nó **không nhìn thấy `w`**.

Cell dưới so sánh trên vài input.

In [15]:
def max_envelopes_wrong_h_only(envelopes):
    """SAI: sắp (w tăng, h tăng) rồi chỉ LIS trên h."""
    envs = sorted(envelopes)                    # (w tăng, h tăng)
    return lis_dp([h for _, h in envs])

for envs in ([[1, 1], [1, 2], [1, 3]],
             [[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]],
             [[5, 4], [6, 4], [6, 7], [2, 3]]):
    print(f"{str(envs):42s} chỉ-so-h = {max_envelopes_wrong_h_only(envs)}, "
          f"đúng = {max_envelopes_subsets(envs)}")

[[1, 1], [1, 2], [1, 3]]                   chỉ-so-h = 3, đúng = 1
[[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]]   chỉ-so-h = 5, đúng = 4
[[5, 4], [6, 4], [6, 7], [2, 3]]           chỉ-so-h = 3, đúng = 3


Example 1 tình cờ vẫn ra đúng (nên đừng tin một test!), nhưng hai input kia bị **đếm thừa**: với `[[4,5],[4,6],[6,7],[2,3],[1,1]]`, dãy `h` sau khi sắp là `[1, 3, 5, 6, 7]`, LIS lấy cả `[4,5]` lẫn `[4,6]` (cùng rộng 4) → 5 thay vì 4.

### Cách 2 vướng ở đâu?

Cách 2 đúng, nhưng hai vòng lặp lồng nhau là `~n²/2` phép so sánh. Với `n = 10^5` là `5·10^9` — trên LeetCode chắc chắn **Time Limit Exceeded (TLE)**. Đo thử với `n` nhỏ để cảm nhận: mỗi lần `n` gấp đôi, thời gian gấp khoảng **4 lần**.

In [16]:
import random, time

random.seed(354)
for n in [500, 1000, 2000]:
    envs = [[random.randint(1, 10**5), random.randint(1, 10**5)] for _ in range(n)]
    t0 = time.perf_counter()
    max_envelopes_dp(envs)
    dt = time.perf_counter() - t0
    print(f"n = {n:5d}: {dt:.3f} giây")
print("Ước lượng n = 10^5: gấp (10^5 / 2000)^2 = 2500 lần thời gian của n = 2000 -> quá chậm.")

n =   500: 0.011 giây
n =  1000: 0.043 giây


n =  2000: 0.183 giây
Ước lượng n = 10^5: gấp (10^5 / 2000)^2 = 2500 lần thời gian của n = 2000 -> quá chậm.


## 10. Bước đột phá 2 — Mẹo sắp xếp: `w` tăng, nhưng `h` GIẢM khi `w` bằng nhau

Ta muốn **bỏ hẳn việc so `w`** để chỉ còn LIS một chiều trên `h` (vì LIS một chiều có thuật toán `O(n log n)` ở phần sau). Vấn đề duy nhất ở phần 9 là: LIS trên `h` có thể **chọn hai phong bì cùng `w`**. Làm sao cấm điều đó mà không cần nhìn `w`?

**Mẹo:** sắp theo `w` tăng, và **trong cùng một `w`, sắp `h` giảm dần**:

```python
envs.sort(key=lambda e: (e[0], -e[1]))
```

- `key` trả về **tuple** `(w, -h)`. Python so tuple **từng thành phần một**: so `w` trước; nếu `w` bằng nhau mới so tới `-h`.
- `-h` nhỏ hơn khi `h` lớn hơn, nên `h` lớn đứng trước → `h` **giảm dần** trong nhóm cùng `w`.

**Vì sao mẹo này hiệu quả?** Trong một nhóm cùng `w`, các `h` đứng theo thứ tự **giảm** (hoặc bằng), ví dụ nhóm `w = 1`: `[1,3], [1,2], [1,1]` → dãy `h` là `3, 2, 1`. Một dãy con **tăng hẳn** không thể lấy **hai** phần tử từ đoạn `3, 2, 1` này (đi tới thì chỉ giảm hoặc bằng, không bao giờ tăng). Vậy LIS trên `h` **tự động chọn tối đa một phong bì mỗi nhóm `w`** — đúng điều ta cần, mà không phải nhìn `w`!

Thử lại phản ví dụ `[[1,1],[1,2],[1,3]]`: sắp theo `(w, -h)` → `[1,3],[1,2],[1,1]`, dãy `h = [3, 2, 1]`, LIS = **1** ✓.

In [17]:
envs = [[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]]

sort_h_up = sorted(envs)                                 # (w tăng, h tăng)
sort_h_down = sorted(envs, key=lambda e: (e[0], -e[1]))  # (w tăng, h giảm)

print("Sắp (w, h)  :", sort_h_up,   "-> dãy h:", [h for _, h in sort_h_up])
print("Sắp (w, -h) :", sort_h_down, "-> dãy h:", [h for _, h in sort_h_down])
print()
print("LIS trên h khi h tăng :", lis_dp([h for _, h in sort_h_up]),   "(SAI, đếm cả [4,5] và [4,6])")
print("LIS trên h khi h giảm :", lis_dp([h for _, h in sort_h_down]), "(ĐÚNG)")
print("Trọng tài             :", max_envelopes_subsets(envs))

# Tuple được so từng thành phần:
print((4, -6) < (4, -5), (2, -3) < (4, -6))

Sắp (w, h)  : [[1, 1], [2, 3], [4, 5], [4, 6], [6, 7]] -> dãy h: [1, 3, 5, 6, 7]
Sắp (w, -h) : [[1, 1], [2, 3], [4, 6], [4, 5], [6, 7]] -> dãy h: [1, 3, 6, 5, 7]

LIS trên h khi h tăng : 5 (SAI, đếm cả [4,5] và [4,6])
LIS trên h khi h giảm : 4 (ĐÚNG)
Trọng tài             : 4
True True


### Chứng minh mẹo đúng theo cả hai chiều

Sau khi sắp theo `(w tăng, h giảm)`, gọi dãy chiều cao là `H`. Ta cần: **độ dài chuỗi lồng dài nhất = LIS (tăng hẳn) của `H`**. Chứng minh hai chiều:

**(a) Mỗi chuỗi lồng nhau → một dãy con tăng hẳn của `H` cùng độ dài.** Chuỗi lồng `e1 → e2 → ... → ek` có `w` tăng hẳn, nên trong list đã sắp chúng đứng đúng thứ tự trái → phải (vì sắp theo `w` trước). Và `h` của chúng tăng hẳn. Vậy các `h` đó tạo một dãy con tăng hẳn của `H`. ⇒ LIS ≥ đáp án.

**(b) Mỗi dãy con tăng hẳn của `H` → một chuỗi lồng nhau hợp lệ.** Lấy hai phần tử liền nhau trong dãy con: vị trí `i < j`, `H[i] < H[j]`. Có hai khả năng cho `w`:
- Nếu `w_i = w_j` (cùng nhóm): trong nhóm `h` giảm dần nên `H[i] >= H[j]` — **mâu thuẫn**. Vậy không thể xảy ra.
- Vậy `w_i < w_j` (sắp tăng theo `w`), cộng với `h_i < h_j` ⇒ `fits(e_i, e_j)` ✓.

⇒ đáp án ≥ LIS. Hai chiều gộp lại: **đáp án = LIS**.

**Chú ý chữ "hẳn" ở LIS:** phải là LIS **tăng hẳn** (`<`), không phải "không giảm" (`<=`). Phản ví dụ: `[[1,3],[2,3]]` — khác `w` nhưng **cùng `h = 3`**, không lồng được (đáp án 1). Dãy `H = [3, 3]`: LIS tăng hẳn = 1 ✓, còn dãy con không giảm dài nhất = 2 ✗.

In [18]:
def max_envelopes_sort_trick_dp(envelopes):
    """Cách 3: mẹo sort (w tăng, h giảm) + LIS 1 chiều trên h bằng DP O(n²)."""
    envs = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    return lis_dp([h for _, h in envs])

tests = [
    [[5, 4], [6, 4], [6, 7], [2, 3]],
    [[1, 1], [1, 1], [1, 1]],
    [[1, 1], [1, 2], [1, 3]],
    [[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]],
    [[1, 3], [2, 3]],
]
for envs in tests:
    print(f"{str(envs):42s} cách 3 = {max_envelopes_sort_trick_dp(envs)}, "
          f"trọng tài = {max_envelopes_subsets(envs)}")

[[5, 4], [6, 4], [6, 7], [2, 3]]           cách 3 = 3, trọng tài = 3
[[1, 1], [1, 1], [1, 1]]                   cách 3 = 1, trọng tài = 1
[[1, 1], [1, 2], [1, 3]]                   cách 3 = 1, trọng tài = 1
[[4, 5], [4, 6], [6, 7], [2, 3], [1, 1]]   cách 3 = 4, trọng tài = 4
[[1, 3], [2, 3]]                           cách 3 = 1, trọng tài = 1


Cách 3 vẫn `O(n²)` (vì `lis_dp` là `O(n²)`), **nhưng** bài toán đã được rút gọn thành **LIS thuần một chiều trên số nguyên**. Bây giờ chỉ còn một việc: giải LIS nhanh hơn. Để làm vậy ta cần công cụ **tìm kiếm nhị phân**.

## 11. Kiến thức nền: tìm kiếm nhị phân (binary search) và module `bisect`

**Bài toán:** cho một list **đã sắp tăng** `a` và một số `x`, tìm **vị trí đầu tiên `k` sao cho `a[k] >= x`** (nếu mọi phần tử đều `< x` thì `k = len(a)`). Đây cũng chính là **chỗ có thể chèn `x` vào mà list vẫn sắp xếp**, chèn vào **bên trái** các phần tử bằng `x`.

Duyệt từ đầu mất `O(n)`. **Tìm kiếm nhị phân** làm trong `O(log n)`: nhìn phần tử ở **giữa**, nếu nó `< x` thì câu trả lời nằm ở **nửa phải**, ngược lại nằm ở **nửa trái** (kể cả chính nó). Mỗi bước loại một nửa → `10^5` phần tử chỉ cần ~17 bước.

Python có sẵn module `bisect`:

- `bisect_left(a, x)`: vị trí đầu tiên có `a[k] >= x` (chèn **trước** các số bằng `x`).
- `bisect_right(a, x)`: vị trí đầu tiên có `a[k] > x` (chèn **sau** các số bằng `x`).

Hai hàm chỉ khác nhau khi `x` **có mặt** trong list — và khác biệt đó chính là "tăng hẳn" vs "không giảm" (sẽ quan trọng ở phần lỗi thường gặp).

In [19]:
from bisect import bisect_left, bisect_right

a = [1, 3, 3, 3, 7, 9]
for x in [0, 3, 4, 9, 10]:
    print(f"x = {x:2d}: bisect_left = {bisect_left(a, x)}, bisect_right = {bisect_right(a, x)}")

x =  0: bisect_left = 0, bisect_right = 0
x =  3: bisect_left = 1, bisect_right = 4
x =  4: bisect_left = 4, bisect_right = 4
x =  9: bisect_left = 5, bisect_right = 6
x = 10: bisect_left = 6, bisect_right = 6


### Tự cài `bisect_left` để hiểu nó hoạt động thế nào

Giữ một khoảng tìm kiếm `[lo, hi)` (nửa mở: gồm `lo`, không gồm `hi`) mà ta **chắc chắn** đáp án nằm trong đó, ban đầu `lo = 0, hi = len(a)`:

```text
while lo < hi:
    mid = (lo + hi) // 2
    if a[mid] < x:   lo = mid + 1     # a[mid] quá nhỏ, đáp án ở bên phải mid
    else:            hi = mid         # a[mid] >= x, mid có thể là đáp án -> giữ lại
return lo
```

Bất biến (invariant): mọi vị trí `< lo` đều có giá trị `< x`; mọi vị trí `>= hi` đều có giá trị `>= x`. Khi `lo == hi`, đó chính là vị trí đầu tiên `>= x`.

In [20]:
def my_bisect_left(a, x, verbose=False):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if verbose:
            print(f"  lo={lo}, hi={hi}, mid={mid}, a[mid]={a[mid]} {'<' if a[mid] < x else '>='} {x}")
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [1, 3, 3, 3, 7, 9]
print("Tìm x = 4:")
print("->", my_bisect_left(a, 4, verbose=True))

random.seed(1)
for _ in range(1000):
    arr = sorted(random.randint(0, 20) for _ in range(random.randint(0, 10)))
    x = random.randint(-1, 21)
    assert my_bisect_left(arr, x) == bisect_left(arr, x)
print("my_bisect_left khớp bisect_left trên 1000 test ngẫu nhiên")

Tìm x = 4:
  lo=0, hi=6, mid=3, a[mid]=3 < 4
  lo=4, hi=6, mid=5, a[mid]=9 >= 4
  lo=4, hi=5, mid=4, a[mid]=7 >= 4
-> 4
my_bisect_left khớp bisect_left trên 1000 test ngẫu nhiên


## 12. Bước đột phá 3 — Patience sorting: LIS bằng mảng `tails`

**Trò chơi xếp bài (patience sorting).** Lần lượt lật từng lá bài (từng số `x`), đặt lên các **chồng** theo luật: đặt `x` lên **chồng trái nhất** có lá trên cùng `>= x`; nếu không có chồng nào như vậy thì **mở chồng mới bên phải**. Kết thúc, **số chồng = độ dài LIS**.

Ta không cần nhớ cả chồng, chỉ cần nhớ **lá trên cùng của mỗi chồng**. Gọi list đó là `tails`. Nó có một ý nghĩa rất đẹp:

> **`tails[k]` = giá trị cuối NHỎ NHẤT có thể có của một dãy con tăng hẳn dài `k + 1`** (trong phần dãy đã duyệt).

**Tại sao lại quan tâm "đuôi nhỏ nhất"?** Vì đuôi càng nhỏ thì càng **dễ nối dài thêm**. Ví dụ đã có hai dãy dài 2 là `[2, 5]` và `[2, 3]`: nếu tiếp theo là số `4`, chỉ `[2, 3]` nối được thành `[2, 3, 4]`. Nên trong các dãy cùng độ dài, chỉ cần nhớ cái có đuôi nhỏ nhất.

**Luật cập nhật với mỗi số `x`:**

1. `k = bisect_left(tails, x)` — vị trí đầu tiên có `tails[k] >= x`.
2. Nếu `k == len(tails)`: `x` lớn hơn mọi đuôi → nối được vào dãy dài nhất → `tails.append(x)` (LIS dài thêm 1).
3. Ngược lại: `x` nối được vào dãy dài `k` (vì `tails[k-1] < x`), tạo dãy dài `k + 1` có đuôi `x <= tails[k]` → **đuôi tốt hơn**, nên `tails[k] = x`.

Đáp án = `len(tails)`. Mỗi số tốn một `bisect` `O(log n)` → tổng `O(n log n)`.

Cell dưới mô phỏng các **chồng bài** thật (để thấy `tails` là các lá trên cùng).

In [21]:
def patience_piles(nums):
    piles = []                                  # mỗi chồng là 1 list, lá trên cùng ở cuối
    for x in nums:
        tops = [p[-1] for p in piles]           # các lá trên cùng = tails, luôn tăng dần
        k = bisect_left(tops, x)
        if k == len(piles):
            piles.append([x])
        else:
            piles[k].append(x)
        print(f"lật {x:3d} -> chồng {k}: {piles}")
    return len(piles)

print("Số chồng = LIS =", patience_piles([10, 9, 2, 5, 3, 7, 101, 18]))

lật  10 -> chồng 0: [[10]]
lật   9 -> chồng 0: [[10, 9]]
lật   2 -> chồng 0: [[10, 9, 2]]
lật   5 -> chồng 1: [[10, 9, 2], [5]]
lật   3 -> chồng 1: [[10, 9, 2], [5, 3]]
lật   7 -> chồng 2: [[10, 9, 2], [5, 3], [7]]
lật 101 -> chồng 3: [[10, 9, 2], [5, 3], [7], [101]]
lật  18 -> chồng 3: [[10, 9, 2], [5, 3], [7], [101, 18]]
Số chồng = LIS = 4


### Code LIS `O(n log n)` và bảng trace `tails`

Bảng dưới là trace tay cho `nums = [10, 9, 2, 5, 3, 7, 101, 18]` (cột "Hành động" theo đúng luật 1–3 ở trên). Cell kế tiếp in ra đúng các trạng thái này.

| i | x | `tails` trước | k | Hành động | `tails` sau |
|---:|---:|---|---:|---|---|
| 0 | `10` | `[]` | `0` | append (dãy dài thêm 1) | `[10]` |
| 1 | `9` | `[10]` | `0` | thay tails[0]=10 → 9 | `[9]` |
| 2 | `2` | `[9]` | `0` | thay tails[0]=9 → 2 | `[2]` |
| 3 | `5` | `[2]` | `1` | append (dãy dài thêm 1) | `[2, 5]` |
| 4 | `3` | `[2, 5]` | `1` | thay tails[1]=5 → 3 | `[2, 3]` |
| 5 | `7` | `[2, 3]` | `2` | append (dãy dài thêm 1) | `[2, 3, 7]` |
| 6 | `101` | `[2, 3, 7]` | `3` | append (dãy dài thêm 1) | `[2, 3, 7, 101]` |
| 7 | `18` | `[2, 3, 7, 101]` | `3` | thay tails[3]=101 → 18 | `[2, 3, 7, 18]` |

Kết quả `len(tails) = 4` ✓. Để ý bước `x = 3`: nó **thay** `5` ở `tails[1]` — dãy dài 2 tốt nhất giờ là `[2, 3]` thay vì `[2, 5]`. Bước `x = 18` thay `101`: dãy dài 4 tốt nhất kết thúc bằng 18 thay vì 101.

In [22]:
def lis_tails(nums):
    """LIS tăng hẳn trong O(n log n)."""
    tails = []
    for x in nums:
        k = bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

def trace_lis_tails(nums):
    tails = []
    for i, x in enumerate(nums):
        k = bisect_left(tails, x)
        before = list(tails)
        if k == len(tails):
            tails.append(x)
            act = "append"
        else:
            act = f"thay tails[{k}]={tails[k]}"
            tails[k] = x
        print(f"i={i}, x={x:3d}: tails trước={str(before):18s} k={k}  {act:15s} -> tails={tails}")
    print("LIS =", len(tails))

trace_lis_tails([10, 9, 2, 5, 3, 7, 101, 18])
print(lis_tails([0, 1, 0, 3, 2, 3]), lis_tails([7, 7, 7, 7]))   # 4 1 (2 ví dụ khác của LeetCode 300)

i=0, x= 10: tails trước=[]                 k=0  append          -> tails=[10]
i=1, x=  9: tails trước=[10]               k=0  thay tails[0]=10 -> tails=[9]
i=2, x=  2: tails trước=[9]                k=0  thay tails[0]=9 -> tails=[2]
i=3, x=  5: tails trước=[2]                k=1  append          -> tails=[2, 5]
i=4, x=  3: tails trước=[2, 5]             k=1  thay tails[1]=5 -> tails=[2, 3]
i=5, x=  7: tails trước=[2, 3]             k=2  append          -> tails=[2, 3, 7]
i=6, x=101: tails trước=[2, 3, 7]          k=3  append          -> tails=[2, 3, 7, 101]
i=7, x= 18: tails trước=[2, 3, 7, 101]     k=3  thay tails[3]=101 -> tails=[2, 3, 7, 18]
LIS = 4
4 1


### Cảnh báo: `tails` KHÔNG phải là một dãy con thật

Rất nhiều người tưởng cuối cùng `tails` chính là LIS. **Sai!** `tails` chỉ đúng về **độ dài**. Mỗi ô `tails[k]` là đuôi tốt nhất của dãy dài `k+1`, nhưng các ô có thể đến từ **những dãy khác nhau**.

Ví dụ `nums = [3, 4, 5, 1]`:

- Sau `3, 4, 5`: `tails = [3, 4, 5]` (đây đúng là một dãy thật).
- Lật `1`: `bisect_left = 0`, thay `3` → `tails = [1, 4, 5]`.

`[1, 4, 5]` **không** phải dãy con của `[3, 4, 5, 1]` (số `1` đứng **sau** `5`). Nhưng độ dài 3 vẫn đúng: LIS thật là `[3, 4, 5]`. Ô `tails[0] = 1` chỉ có nghĩa "nếu sau này có số lớn hơn 1, một dãy dài 1 kết thúc bằng 1 sẵn sàng được nối".

(Với `[10, 9, 2, 5, 3, 7, 101, 18]` thì `tails` cuối `[2, 3, 7, 18]` **tình cờ** là dãy thật — cell dưới kiểm tra cả hai trường hợp. Không bao giờ được **dựa vào** sự tình cờ đó.)

Muốn lấy **dãy thật**, ta lưu thêm: `tails_idx[k]` = **chỉ số** của phần tử đang làm đuôi, và `parent[i]` = chỉ số phần tử đứng ngay trước `i` trong dãy (chính là `tails_idx[k-1]` lúc `i` được đặt vào). Cuối cùng đi ngược từ `tails_idx[-1]` theo `parent`.

In [23]:
def lis_with_sequence(nums):
    tails, tails_idx = [], []
    parent = [-1] * len(nums)
    for i, x in enumerate(nums):
        k = bisect_left(tails, x)
        parent[i] = tails_idx[k - 1] if k > 0 else -1   # phần tử đứng trước x
        if k == len(tails):
            tails.append(x)
            tails_idx.append(i)
        else:
            tails[k] = x
            tails_idx[k] = i
    seq, i = [], tails_idx[-1] if tails_idx else -1
    while i != -1:                       # đi ngược theo parent
        seq.append(nums[i])
        i = parent[i]
    return tails, seq[::-1]

def is_subsequence(sub, nums):
    it = iter(nums)
    return all(any(x == y for y in it) for x in sub)   # tìm lần lượt từng x, không quay lại

for nums in ([3, 4, 5, 1], [10, 9, 2, 5, 3, 7, 101, 18]):
    tails, seq = lis_with_sequence(nums)
    print(f"nums={nums}")
    print(f"   tails = {tails}, là dãy con thật? {is_subsequence(tails, nums)}")
    print(f"   LIS thật khôi phục bằng parent = {seq}, là dãy con thật? {is_subsequence(seq, nums)}")

nums=[3, 4, 5, 1]
   tails = [1, 4, 5], là dãy con thật? False
   LIS thật khôi phục bằng parent = [3, 4, 5], là dãy con thật? True
nums=[10, 9, 2, 5, 3, 7, 101, 18]
   tails = [2, 3, 7, 18], là dãy con thật? True
   LIS thật khôi phục bằng parent = [2, 3, 7, 18], là dãy con thật? True


### Vì sao `tails` luôn đúng? (invariant)

**Bất biến:** sau khi duyệt xong một phần dãy, với mọi `k`, `tails[k]` = đuôi nhỏ nhất trong mọi dãy con tăng hẳn dài `k+1` của phần đó, và `tails` **tăng hẳn**.

- **Vì sao `tails` tăng hẳn?** Một dãy dài `k+2` có đuôi `t`; bỏ đi phần tử cuối được dãy dài `k+1` với đuôi `< t`. Vậy đuôi nhỏ nhất của độ dài `k+1` `<` đuôi nhỏ nhất của độ dài `k+2`. Nhờ `tails` tăng, ta mới **được phép** dùng `bisect`.
- **Khi thêm `x`, tại sao chỉ sửa đúng ô `k = bisect_left(tails, x)`?**
  - Với mọi `j < k`: `tails[j] < x`, nên `x` không làm đuôi dãy dài `j+1` tốt hơn được (đuôi `x` lớn hơn) → giữ nguyên.
  - Ô `k`: `tails[k-1] < x` nên nối `x` vào dãy dài `k` được dãy dài `k+1` đuôi `x`; và `x <= tails[k]` → cập nhật (nếu bằng thì không đổi gì).
  - Với mọi `j > k`: muốn `x` làm đuôi dãy dài `j+1` thì phần trước phải là dãy dài `j` có đuôi `< x`, nhưng mọi dãy dài `j` có đuôi `>= tails[j-1] >= tails[k] >= x` → không thể → giữ nguyên.
- Độ dài LIS = số độ dài đang tồn tại = `len(tails)`.

Cell dưới kiểm tra `tails` luôn tăng hẳn và `len(tails)` khớp DP `O(n²)` trên 300 dãy ngẫu nhiên.

In [24]:
random.seed(300)
for _ in range(300):
    nums = [random.randint(1, 8) for _ in range(random.randint(1, 12))]
    tails = []
    for x in nums:
        k = bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
        assert all(tails[i] < tails[i + 1] for i in range(len(tails) - 1))   # luôn tăng hẳn
    assert len(tails) == lis_dp(nums) == lis_bruteforce(nums)
print("Bất biến 'tails tăng hẳn' giữ đúng; lis_tails == lis_dp == brute trên 300 dãy")

Bất biến 'tails tăng hẳn' giữ đúng; lis_tails == lis_dp == brute trên 300 dãy


## 13. Cách 4 (cách chính) — Mẹo sort + LIS `O(n log n)` trên `h`

Ghép hai bước đột phá:

1. Sắp `envelopes` theo `(w tăng, h giảm)`: `envelopes.sort(key=lambda e: (e[0], -e[1]))`.
2. Lấy dãy chiều cao `h` theo thứ tự đó, chạy LIS **tăng hẳn** bằng `tails` + `bisect_left`.
3. Trả về `len(tails)`.

Chỉ vài dòng code, nhưng mỗi dòng gánh một ý quan trọng:

- `(e[0], -e[1])` → loại các cặp cùng rộng (phần 10).
- `bisect_left` (không phải `bisect_right`) → "tăng **hẳn**" cho `h`, loại các cặp cùng cao.
- Chỉ đưa **`h`** vào `tails` (không đưa cả cặp `[w, h]`).

In [25]:
def max_envelopes_lis(envelopes):
    """Cách chính: sort (w tăng, h giảm) + LIS tăng hẳn trên h, O(n log n)."""
    envs = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    tails = []
    for _, h in envs:
        k = bisect_left(tails, h)
        if k == len(tails):
            tails.append(h)
        else:
            tails[k] = h
    return len(tails)

print(max_envelopes_lis([[5, 4], [6, 4], [6, 7], [2, 3]]))   # 3
print(max_envelopes_lis([[1, 1], [1, 1], [1, 1]]))           # 1
print(max_envelopes_lis([[1, 1], [1, 2], [1, 3]]))           # 1
print(max_envelopes_lis([[1, 3], [2, 3]]))                   # 1

3
1
1
1


### Trace Cách 4 trên một input "khó"

Input: `envelopes = [[4,5],[4,6],[6,7],[2,3],[1,1],[5,4],[6,5],[7,2],[8,3]]`

Input này có đủ các tình huống đáng chú ý: hai nhóm cùng rộng (`w = 4` và `w = 6`), phong bì thấp xuất hiện muộn (`[7,2]`, `[8,3]`) làm `tails` bị thay ở giữa.

Sau khi sắp theo `(w tăng, h giảm)`: `[[1,1],[2,3],[4,6],[4,5],[5,4],[6,7],[6,5],[7,2],[8,3]]`

| Bước | Phong bì (sau sort) | h | `tails` trước | `k = bisect_left` | Hành động | `tails` sau |
|---:|---|---:|---|---:|---|---|
| 1 | `[1,1]` | `1` | `[]` | `0` | `1` lớn hơn mọi đuôi → **append** | `[1]` |
| 2 | `[2,3]` | `3` | `[1]` | `1` | `3` lớn hơn mọi đuôi → **append** | `[1, 3]` |
| 3 | `[4,6]` | `6` | `[1, 3]` | `2` | `6` lớn hơn mọi đuôi → **append** | `[1, 3, 6]` |
| 4 | `[4,5]` | `5` | `[1, 3, 6]` | `2` | thay `tails[2] = 6` bằng `5` | `[1, 3, 5]` |
| 5 | `[5,4]` | `4` | `[1, 3, 5]` | `2` | thay `tails[2] = 5` bằng `4` | `[1, 3, 4]` |
| 6 | `[6,7]` | `7` | `[1, 3, 4]` | `3` | `7` lớn hơn mọi đuôi → **append** | `[1, 3, 4, 7]` |
| 7 | `[6,5]` | `5` | `[1, 3, 4, 7]` | `3` | thay `tails[3] = 7` bằng `5` | `[1, 3, 4, 5]` |
| 8 | `[7,2]` | `2` | `[1, 3, 4, 5]` | `1` | thay `tails[1] = 3` bằng `2` | `[1, 2, 4, 5]` |
| 9 | `[8,3]` | `3` | `[1, 2, 4, 5]` | `2` | thay `tails[2] = 4` bằng `3` | `[1, 2, 3, 5]` |

Đáp án = `len(tails) = 4`. Những điểm cần nhìn kỹ:

- **Nhóm `w = 4`**: `[4,6]` đứng trước `[4,5]` (h giảm). `[4,6]` được append, rồi `[4,5]` chỉ **thay** ô đó chứ không nối dài thêm → không bao giờ dùng cả hai phong bì rộng 4. Nhóm `w = 6` cũng vậy (`[6,7]` append rồi `[6,5]` thay).
- **Cuối cùng `tails = [1, 2, 3, 5]`**, đọc như các chiều cao `1, 2, 3, 5` — nhưng phong bì cao 2 và 3 (`[7,2]`, `[8,3]`) đứng **sau** phong bì cao 5 (`[6,5]`), nên đây **không** phải chuỗi thật. Chỉ độ dài 4 là đúng; một chuỗi thật là `[1,1] → [2,3] → [5,4] → [6,5]`.

In [26]:
def trace_envelopes_lis(envelopes, key=lambda e: (e[0], -e[1])):
    envs = sorted(envelopes, key=key)
    print("Sau khi sắp:", envs)
    tails = []
    for step, (w, h) in enumerate(envs, 1):
        k = bisect_left(tails, h)
        before = list(tails)
        if k == len(tails):
            tails.append(h)
            act = "append"
        else:
            act = f"thay tails[{k}]={tails[k]}"
            tails[k] = h
        print(f"bước {step}: [{w},{h}] h={h}  tails trước={str(before):14s} k={k}  {act:14s} -> {tails}")
    print("Đáp án =", len(tails))
    return len(tails)

ENVS_TRACE = [[4, 5], [4, 6], [6, 7], [2, 3], [1, 1], [5, 4], [6, 5], [7, 2], [8, 3]]
trace_envelopes_lis(ENVS_TRACE)
print("Trọng tài:", max_envelopes_subsets(ENVS_TRACE))

Sau khi sắp: [[1, 1], [2, 3], [4, 6], [4, 5], [5, 4], [6, 7], [6, 5], [7, 2], [8, 3]]
bước 1: [1,1] h=1  tails trước=[]             k=0  append         -> [1]
bước 2: [2,3] h=3  tails trước=[1]            k=1  append         -> [1, 3]
bước 3: [4,6] h=6  tails trước=[1, 3]         k=2  append         -> [1, 3, 6]
bước 4: [4,5] h=5  tails trước=[1, 3, 6]      k=2  thay tails[2]=6 -> [1, 3, 5]
bước 5: [5,4] h=4  tails trước=[1, 3, 5]      k=2  thay tails[2]=5 -> [1, 3, 4]
bước 6: [6,7] h=7  tails trước=[1, 3, 4]      k=3  append         -> [1, 3, 4, 7]
bước 7: [6,5] h=5  tails trước=[1, 3, 4, 7]   k=3  thay tails[3]=7 -> [1, 3, 4, 5]
bước 8: [7,2] h=2  tails trước=[1, 3, 4, 5]   k=1  thay tails[1]=3 -> [1, 2, 4, 5]
bước 9: [8,3] h=3  tails trước=[1, 2, 4, 5]   k=2  thay tails[2]=4 -> [1, 2, 3, 5]
Đáp án = 4
Trọng tài: 4


### Demo SAI: cùng thuật toán `tails` nhưng sắp `h` TĂNG khi `w` bằng nhau

Bây giờ chạy đúng hàm trace trên, chỉ đổi `key` thành `(w, h)` (h tăng). Nhóm `w = 4` giờ đứng thành `[4,5], [4,6]`: `5` được append, rồi `6 > 5` cũng được **append** → `tails` dài thêm nhờ hai phong bì **cùng rộng** — điều cấm. Nhóm `w = 6` cũng vậy. Kết quả bị **đếm thừa**.

In [27]:
wrong = trace_envelopes_lis(ENVS_TRACE, key=lambda e: (e[0], e[1]))
print("-> SAI! Đáp án đúng là", max_envelopes_subsets(ENVS_TRACE))
print()
trace_envelopes_lis([[1, 1], [1, 2], [1, 3]], key=lambda e: (e[0], e[1]))
print("-> SAI! Ba phong bì cùng rộng 1, đáp án đúng là 1")

Sau khi sắp: [[1, 1], [2, 3], [4, 5], [4, 6], [5, 4], [6, 5], [6, 7], [7, 2], [8, 3]]
bước 1: [1,1] h=1  tails trước=[]             k=0  append         -> [1]
bước 2: [2,3] h=3  tails trước=[1]            k=1  append         -> [1, 3]
bước 3: [4,5] h=5  tails trước=[1, 3]         k=2  append         -> [1, 3, 5]
bước 4: [4,6] h=6  tails trước=[1, 3, 5]      k=3  append         -> [1, 3, 5, 6]
bước 5: [5,4] h=4  tails trước=[1, 3, 5, 6]   k=2  thay tails[2]=5 -> [1, 3, 4, 6]
bước 6: [6,5] h=5  tails trước=[1, 3, 4, 6]   k=3  thay tails[3]=6 -> [1, 3, 4, 5]
bước 7: [6,7] h=7  tails trước=[1, 3, 4, 5]   k=4  append         -> [1, 3, 4, 5, 7]
bước 8: [7,2] h=2  tails trước=[1, 3, 4, 5, 7] k=1  thay tails[1]=3 -> [1, 2, 4, 5, 7]
bước 9: [8,3] h=3  tails trước=[1, 2, 4, 5, 7] k=2  thay tails[2]=4 -> [1, 2, 3, 5, 7]
Đáp án = 5
-> SAI! Đáp án đúng là 4

Sau khi sắp: [[1, 1], [1, 2], [1, 3]]
bước 1: [1,1] h=1  tails trước=[]             k=0  append         -> [1]
bước 2: [1,2] h=2  tails trước=

### Bonus: in ra chuỗi phong bì thật

LeetCode chỉ hỏi **độ dài**, nhưng phỏng vấn viên hay hỏi thêm "chuỗi đó là những phong bì nào?". Dùng đúng kỹ thuật `tails_idx` + `parent` ở phần 12, áp dụng lên list đã sắp. Ta kiểm tra chuỗi in ra bằng `is_valid_chain`.

In [28]:
def max_envelopes_with_chain(envelopes):
    envs = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    tails, tails_idx = [], []
    parent = [-1] * len(envs)
    for i, (_, h) in enumerate(envs):
        k = bisect_left(tails, h)
        parent[i] = tails_idx[k - 1] if k > 0 else -1
        if k == len(tails):
            tails.append(h)
            tails_idx.append(i)
        else:
            tails[k] = h
            tails_idx[k] = i
    chain, i = [], tails_idx[-1]
    while i != -1:
        chain.append(envs[i])
        i = parent[i]
    return len(tails), chain[::-1]

for envs in ([[5, 4], [6, 4], [6, 7], [2, 3]], ENVS_TRACE):
    length, chain = max_envelopes_with_chain(envs)
    print(f"độ dài = {length}, chuỗi = {chain}, hợp lệ? {is_valid_chain(chain)}")

độ dài = 3, chuỗi = [[2, 3], [5, 4], [6, 7]], hợp lệ? True
độ dài = 4, chuỗi = [[1, 1], [2, 3], [5, 4], [6, 5]], hợp lệ? True


## 14. Code nộp LeetCode — Cách 4

Bản nộp gọn, không `print`. Trên LeetCode, `List` đã được import sẵn; trong notebook ta tự `from typing import List`.

Hai chi tiết nhỏ cho bản nộp:

- Dùng `envelopes.sort(...)` (sắp tại chỗ) để khỏi tạo list mới — LeetCode cho phép sửa input.
- Viết `tails[k] = h` cho cả hai trường hợp bằng mẹo: nếu `k == len(tails)` thì `append`, ngược lại gán.

### Bản tham khảo — Cách 2 dạng `class Solution` (sẽ TLE)

Để thấy rõ khác biệt, đây là Cách 2 (DP `O(n²)`) viết đúng format LeetCode. Nó **đúng** trên mọi test nhỏ, nhưng với `n = 10^5` sẽ bị **Time Limit Exceeded**. Đặt tên `SolutionDP` để không ghi đè bản chính.

In [29]:
from typing import List
from bisect import bisect_left


class Solution:
    def maxEnvelopes(self, envelopes: List[List[int]]) -> int:
        # w tăng; cùng w thì h giảm -> LIS trên h không chọn được 2 phong bì cùng rộng
        envelopes.sort(key=lambda e: (e[0], -e[1]))
        tails = []                      # tails[k] = chiều cao cuối nhỏ nhất của chuỗi dài k+1
        for _, h in envelopes:
            k = bisect_left(tails, h)   # bisect_left -> tăng HẲN
            if k == len(tails):
                tails.append(h)
            else:
                tails[k] = h
        return len(tails)


sol = Solution()
assert sol.maxEnvelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3          # Example 1
assert sol.maxEnvelopes([[1, 1], [1, 1], [1, 1]]) == 1                  # Example 2
assert sol.maxEnvelopes([[7, 8]]) == 1                                  # n = 1
assert sol.maxEnvelopes([[1, 1], [1, 2], [1, 3]]) == 1                  # cùng rộng
assert sol.maxEnvelopes([[1, 3], [2, 3], [3, 3]]) == 1                  # cùng cao
assert sol.maxEnvelopes([[3, 2], [2, 3]]) == 1                          # không được xoay
assert sol.maxEnvelopes([[k, k] for k in range(1, 101)]) == 100         # lồng hết
assert sol.maxEnvelopes([[k, 101 - k] for k in range(1, 101)]) == 1     # w tăng, h giảm hết
assert sol.maxEnvelopes([[4, 5], [4, 6], [6, 7], [2, 3], [1, 1], [5, 4], [6, 5], [7, 2], [8, 3]]) == 4
assert sol.maxEnvelopes([[2, 100], [3, 200], [4, 300], [5, 500], [5, 400], [5, 250], [6, 370], [6, 360], [7, 380]]) == 5
print("Tất cả test của class Solution đều PASS!")


class SolutionDP:
    def maxEnvelopes(self, envelopes: List[List[int]]) -> int:
        envelopes = sorted(envelopes, key=lambda e: (e[0], -e[1]))
        hs = [h for _, h in envelopes]
        dp = [1] * len(hs)
        for i in range(len(hs)):
            for j in range(i):
                if hs[j] < hs[i]:           # nhờ mẹo sort, chỉ cần so h
                    dp[i] = max(dp[i], dp[j] + 1)
        return max(dp)

assert SolutionDP().maxEnvelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
assert SolutionDP().maxEnvelopes([[1, 1], [1, 1], [1, 1]]) == 1
print("SolutionDP PASS (nhưng O(n²) — chỉ dùng tham khảo)")

Tất cả test của class Solution đều PASS!
SolutionDP PASS (nhưng O(n²) — chỉ dùng tham khảo)


## 15. Test đối chiếu chéo tất cả các cách

Sinh ngẫu nhiên nhiều bộ phong bì nhỏ (`n ≤ 9`, `w, h ∈ [1, 6]` để có **nhiều trùng rộng/trùng cao** — đúng chỗ dễ sai), với `random.seed` cố định để kết quả lặp lại được. So 6 cài đặt với **trọng tài** `max_envelopes_subsets`. Ta cũng kiểm tra chuỗi do `max_envelopes_with_chain` trả về là hợp lệ và đúng độ dài.

### Kiểm tra hiệu năng ở kích thước thật của đề

`n = 10^5`, `w, h` ngẫu nhiên trong `[1, 10^5]`, cộng thêm hai trường hợp đặc biệt: tất cả lồng được vào nhau (đáp án `n`), và tất cả cùng rộng (đáp án 1). Cách 4 chạy trong khoảng một phần nhỏ của giây (thời gian cụ thể tùy máy).

In [30]:
random.seed(2026)
methods = {
    "dfs_memo": max_envelopes_dfs_memo,
    "dp_O(n²)": max_envelopes_dp,
    "sort_trick_dp": max_envelopes_sort_trick_dp,
    "lis_tails": max_envelopes_lis,
    "Solution": lambda e: Solution().maxEnvelopes([list(x) for x in e]),
    "SolutionDP": lambda e: SolutionDP().maxEnvelopes(e),
}
for t in range(500):
    n = random.randint(1, 9)
    envs = [[random.randint(1, 6), random.randint(1, 6)] for _ in range(n)]
    expected = max_envelopes_subsets(envs)
    for name, f in methods.items():
        got = f([list(x) for x in envs])
        assert got == expected, (name, envs, got, expected)
    length, chain = max_envelopes_with_chain(envs)
    assert length == expected == len(chain) and is_valid_chain(chain)

# DFS không memo (mũ) chỉ test với n rất nhỏ
for t in range(200):
    envs = [[random.randint(1, 5), random.randint(1, 5)] for _ in range(random.randint(1, 7))]
    assert max_envelopes_dfs(envs) == max_envelopes_subsets(envs)
print("500 + 200 test ngẫu nhiên: mọi cách khớp trọng tài")


random.seed(10**5)
big_random = [[random.randint(1, 10**5), random.randint(1, 10**5)] for _ in range(10**5)]
big_chain = [[k, k] for k in range(1, 10**5 + 1)]
random.shuffle(big_chain)
big_same_w = [[7, random.randint(1, 10**5)] for _ in range(10**5)]

for name, envs in [("ngẫu nhiên", big_random), ("lồng hết", big_chain), ("cùng rộng", big_same_w)]:
    t0 = time.perf_counter()
    ans = Solution().maxEnvelopes([list(e) for e in envs])
    dt = time.perf_counter() - t0
    print(f"{name:11s}: đáp án = {ans:6d}, thời gian = {dt:.3f} giây")
assert Solution().maxEnvelopes([list(e) for e in big_chain]) == 10**5
assert Solution().maxEnvelopes([list(e) for e in big_same_w]) == 1

500 + 200 test ngẫu nhiên: mọi cách khớp trọng tài


ngẫu nhiên : đáp án =    606, thời gian = 0.139 giây
lồng hết   : đáp án = 100000, thời gian = 0.116 giây


cùng rộng  : đáp án =      1, thời gian = 0.140 giây


## 16. Vì sao đúng — tóm tắt cho từng cách

- **Cách 1a (tập con):** thử **mọi** nhóm; một nhóm lồng được thì khi sắp theo `w` các cặp liền kề phải lồng được (bắc cầu). Không bỏ sót vì duyệt hết.
- **Cách 1b (DFS + memo):** `longest_from(i)` = 1 + max của các đỉnh `i` đi tới được. Đúng vì chuỗi tốt nhất bắt đầu ở `i` phải đi tiếp sang **một** phong bì `j` chứa `i`, và phần còn lại từ `j` cũng phải tốt nhất (nếu không, thay bằng phần tốt hơn sẽ được chuỗi dài hơn). Đồ thị là DAG nên đệ quy dừng.
- **Cách 2 (sort theo `w` + DP):** sau khi sắp theo `w`, phong bì đứng trước trong chuỗi luôn có vị trí nhỏ hơn. `dp[i]` = chuỗi dài nhất có phong bì **ngoài cùng** là `i`. So **cả hai chiều** nên không chọn nhầm cặp cùng rộng.
- **Cách 3 (mẹo sort + LIS):** chứng minh hai chiều ở phần 10: mỗi chuỗi lồng ↔ một dãy con tăng hẳn của `H`; cùng `w` thì `h` giảm nên dãy tăng hẳn không thể lấy 2 phần tử cùng nhóm.
- **Cách 4 (`tails`):** Cách 3 + bất biến của `tails` (phần 12): `tails[k]` là chiều cao cuối nhỏ nhất của chuỗi dài `k+1`, `tails` tăng hẳn, mỗi bước chỉ sửa đúng ô `bisect_left`. Nên `len(tails)` = LIS = đáp án.

**Câu tóm tắt nhớ lâu:** *"Sắp để bỏ một chiều; sắp ngược chiều kia khi hòa để cấm hòa; LIS tăng hẳn với `bisect_left` cho chiều còn lại."*

## 17. Độ phức tạp và so sánh các cách

Gọi `n` = số phong bì (`n ≤ 10^5`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|---|
| 1a. Tập con | thử mọi nhóm, kiểm tra chuỗi | `O(2^n · n)` | `O(n)` | chỉ làm trọng tài, `n ≤ ~15` |
| 1b. DFS không memo | đường dài nhất trên đồ thị | mũ, tới `O(2^n)` lời gọi | `O(n²)` cho đồ thị | minh họa vì sao cần memo |
| 1b. DFS + memo | như trên + ghi nhớ | `O(n²)` | `O(n²)` (đồ thị) | đúng nhưng tốn bộ nhớ |
| 2. Sort `w` + DP | LIS `O(n²)`, so cả 2 chiều | `O(n log n + n²)` = `O(n²)` | `O(n)` | đúng, TLE với `n = 10^5` |
| 3. Mẹo sort + DP | `(w, -h)` rồi LIS 1 chiều `O(n²)` | `O(n²)` | `O(n)` | bước trung gian, rút gọn bài |
| **4. Mẹo sort + `tails`** | `(w, -h)` rồi LIS `O(n log n)` | **`O(n log n)`** | **`O(n)`** | **cách chính, nộp LeetCode** |

Chi tiết Cách 4: sắp xếp `O(n log n)`; mỗi phong bì một lần `bisect_left` trên `tails` dài tối đa `n` → `O(log n)`, tổng `O(n log n)`. Bộ nhớ: `tails` tối đa `n` phần tử (cộng bộ nhớ của thuật toán sắp xếp của Python, tối đa `O(n)`).

## 18. Các lỗi người mới hay gặp

### Lỗi 1: Sắp `h` tăng khi `w` bằng nhau

```python
envelopes.sort()                                  # SAI: (w tăng, h tăng)
envelopes.sort(key=lambda e: (e[0], -e[1]))       # ĐÚNG: (w tăng, h giảm)
```
**Vì sao sai:** trong nhóm cùng `w`, dãy `h` tăng → LIS chọn nhiều phong bì cùng rộng (đã demo ở phần 13).

### Lỗi 2: Chỉ sắp theo `w`, không quy định `h`

```python
envelopes.sort(key=lambda e: e[0])                # SAI (tùy may rủi)
```
**Vì sao sai:** stable sort giữ thứ tự **input** cho nhóm cùng `w`. Nếu input tình cờ đưa `h` nhỏ trước thì LIS đếm thừa. Code "đúng với Example 1" nhưng sai với input khác.

### Lỗi 3: Dùng `bisect_right` thay `bisect_left`

**Vì sao sai:** `bisect_right` cho `h` bằng nhau được **append** → coi `h` bằng nhau là "tăng" → đếm cặp cùng cao như `[[1,3],[2,3]]`.

### Lỗi 4: `bisect` trên cả cặp `[w, h]` thay vì chỉ `h`

```python
k = bisect_left(tails, [w, h])                    # SAI: so list theo thứ tự từ điển
```
**Vì sao sai:** Python so `[5,1] < [6,0]` là `True` (chỉ nhìn `w` trước), nên một phong bì thấp hơn vẫn bị coi là "lớn hơn". `tails` phải chỉ chứa **`h`**.

### Lỗi 5: Trả về `tails` hoặc tin `tails` là chuỗi thật

Đề hỏi **số lượng**: `return len(tails)`. Và `tails` không phải chuỗi phong bì thật (phần 12, 13).

### Lỗi 6: Quên luật không được xoay

Có người "chuẩn hóa" mỗi phong bì thành `[min, max]`. Sai: `[3,2]` và `[2,3]` là hai phong bì khác nhau, không lồng được vào nhau.

Cell dưới chạy các phiên bản sai trên phản ví dụ để thấy tận mắt.

In [31]:
def wrong_sort_h_up(envs):
    envs = sorted(envs)
    tails = []
    for _, h in envs:
        k = bisect_left(tails, h)
        tails[k:k + 1] = [h]          # gán/append gọn: nếu k == len thì thêm vào cuối
    return len(tails)

def wrong_sort_w_only(envs):
    envs = sorted(envs, key=lambda e: e[0])
    tails = []
    for _, h in envs:
        k = bisect_left(tails, h)
        tails[k:k + 1] = [h]
    return len(tails)

def wrong_bisect_right(envs):
    envs = sorted(envs, key=lambda e: (e[0], -e[1]))
    tails = []
    for _, h in envs:
        k = bisect_right(tails, h)
        tails[k:k + 1] = [h]
    return len(tails)

def wrong_bisect_pairs(envs):
    envs = sorted(envs, key=lambda e: (e[0], -e[1]))
    tails = []
    for e in envs:
        k = bisect_left(tails, e)     # so cả list [w, h]
        tails[k:k + 1] = [e]
    return len(tails)

cases = [
    ("Lỗi 1", wrong_sort_h_up,    [[1, 1], [1, 2], [1, 3]]),
    ("Lỗi 2", wrong_sort_w_only,  [[2, 1], [2, 2]]),
    ("Lỗi 3", wrong_bisect_right, [[1, 3], [2, 3]]),
    ("Lỗi 4", wrong_bisect_pairs, [[5, 9], [6, 1]]),
]
for label, f, envs in cases:
    print(f"{label}: {str(envs):26s} sai = {f(envs)}, đúng = {max_envelopes_lis(envs)}")
print("Lỗi 4 chi tiết: [5,9] < [6,1] ?", [5, 9] < [6, 1])

Lỗi 1: [[1, 1], [1, 2], [1, 3]]   sai = 3, đúng = 1
Lỗi 2: [[2, 1], [2, 2]]           sai = 2, đúng = 1
Lỗi 3: [[1, 3], [2, 3]]           sai = 2, đúng = 1
Lỗi 4: [[5, 9], [6, 1]]           sai = 2, đúng = 1
Lỗi 4 chi tiết: [5,9] < [6,1] ? True


## 19. Mẹo nhận dạng pattern

Khi gặp đề có các dấu hiệu sau, hãy nghĩ ngay tới **"sắp xếp + LIS"**:

1. **"Chuỗi dài nhất / số phần tử nhiều nhất"** mà mỗi phần tử kế tiếp phải **lớn hơn** phần tử trước theo một (hoặc nhiều) tiêu chí → LIS hoặc LIS biến thể.
2. **Thứ tự input không quan trọng** (được tự chọn thứ tự) và có **hai chiều** → **sắp theo chiều thứ nhất** để biến thành dãy, rồi LIS trên chiều thứ hai.
3. Điều kiện **"lớn hơn hẳn"** ở chiều đã sắp → cần **phá hòa** (tie-break) **ngược chiều** ở chiều còn lại (`-h`), để các phần tử bằng nhau ở chiều 1 không thể cùng được chọn.
4. Điều kiện **"lớn hơn hoặc bằng"** (không ngặt) → phá hòa **cùng chiều** và dùng **`bisect_right`** (bài tập 5).
5. `n` tới `10^5` → không chạy được `O(n²)` → LIS bằng `tails` + `bisect`.

**Ghi nhớ bảng chọn `bisect`:**

| Muốn dãy… | Dùng | Ví dụ `tails = [2, 5]`, x = 5 |
|---|---|---|
| tăng hẳn (`<`) | `bisect_left` | k = 1 → thay 5 bằng 5, độ dài giữ 2 |
| không giảm (`<=`) | `bisect_right` | k = 2 → append, độ dài thành 3 |

## 20. Biến thể và bài liên quan

- **LeetCode 300 — Longest Increasing Subsequence**: bài gốc, chính là `lis_tails`.
- **LeetCode 673 — Number of Longest Increasing Subsequence**: đếm **số** dãy LIS → DP `O(n²)` giữ thêm mảng `count`.
- **LeetCode 646 — Maximum Length of Pair Chain**: cặp `[a, b]` nối được nếu `b < c`; giải bằng tham lam theo `b` hoặc LIS.
- **LeetCode 1691 — Maximum Height by Stacking Cuboids**: hộp 3 chiều, **được xoay**, đặt chồng không ngặt (`<=`) → sắp từng hộp + DP `O(n²)`.
- **LeetCode 1964 — Find the Longest Valid Obstacle Course at Each Position**: LIS **không giảm** tại mỗi vị trí → `tails` + `bisect_right`.
- **LeetCode 1671 — Minimum Number of Removals to Make Mountain Array**: LIS từ trái và từ phải ghép lại.
- **Bài toán "chồng hộp" 3 chiều không xoay (box stacking)**: hai chiều còn lại không thể gộp bằng một mẹo sort, phải DP `O(n²)` (hoặc cấu trúc dữ liệu nâng cao) — xem bài tập 6.

Cell dưới minh họa nhanh **LIS không giảm** bằng `bisect_right` (nền tảng của 1964 và bài tập 5).

In [32]:
def longest_non_decreasing(nums):
    tails = []
    for x in nums:
        k = bisect_right(tails, x)      # cho phép bằng -> bisect_right
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

for nums in ([1, 2, 2, 3], [7, 7, 7, 7], [3, 1, 2, 2, 1, 3]):
    print(f"{nums}: tăng hẳn = {lis_tails(nums)}, không giảm = {longest_non_decreasing(nums)}")

[1, 2, 2, 3]: tăng hẳn = 3, không giảm = 4
[7, 7, 7, 7]: tăng hẳn = 1, không giảm = 4
[3, 1, 2, 2, 1, 3]: tăng hẳn = 3, không giảm = 4


## 21. Bài tập luyện tập

Làm theo thứ tự, từ dễ đến khó. **Đừng xem đáp án** ở cuối notebook trước khi tự làm. Các cell TODO chỉ định nghĩa hàm, assert đang bị comment — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

```python
envs = [[3, 4], [1, 9], [3, 7], [2, 2], [1, 5]]
s = sorted(envs, key=lambda e: (e[0], -e[1]))
print(s)
print([h for _, h in s])
```

Hãy viết ra (a) list `s`, (b) dãy `h`, và (c) đáp án `maxEnvelopes(envs)`.

### Bài 2 — Tính tay bảng `tails`

Cho dãy chiều cao `H = [5, 2, 8, 6, 3, 6, 9, 7]`. Lập bảng giống phần 12: với mỗi `x`, ghi `k = bisect_left(tails, x)`, hành động (append/thay ô nào) và `tails` sau bước đó. LIS bằng bao nhiêu? `tails` cuối cùng có phải một dãy con thật của `H` không?

### Bài 3 — Điền TODO: LIS tăng hẳn bằng `tails`

Hoàn thiện hàm `lis_length_todo(nums)` theo 3 luật ở phần 12. Không được dùng DP `O(n²)`.

In [33]:
def lis_length_todo(nums):
    tails = []
    for x in nums:
        # TODO 1: tìm k = vị trí đầu tiên có tails[k] >= x (dùng bisect_left)
        # TODO 2: nếu k == len(tails): append x
        # TODO 3: ngược lại: thay tails[k] bằng x
        pass
    # TODO 4: trả về độ dài LIS
    raise NotImplementedError

# Bỏ comment sau khi làm xong:
# assert lis_length_todo([10, 9, 2, 5, 3, 7, 101, 18]) == 4
# assert lis_length_todo([0, 1, 0, 3, 2, 3]) == 4
# assert lis_length_todo([7, 7, 7, 7]) == 1

### Bài 4 — Tự viết lại lời giải đầy đủ

Không nhìn phần 14, tự viết `exercise_max_envelopes(envelopes)`: sắp đúng key, LIS trên `h` bằng `bisect_left`, trả về độ dài. Thử cả các phản ví dụ của phần 18.

In [34]:
def exercise_max_envelopes(envelopes):
    # TODO: sắp theo (w tăng, h giảm)
    # TODO: LIS tăng hẳn trên h bằng tails + bisect_left
    raise NotImplementedError

# assert exercise_max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
# assert exercise_max_envelopes([[1, 1], [1, 1], [1, 1]]) == 1
# assert exercise_max_envelopes([[1, 1], [1, 2], [1, 3]]) == 1
# assert exercise_max_envelopes([[1, 3], [2, 3]]) == 1

### Bài 5 — Biến thể: chồng hộp **không ngặt**

Bây giờ luật đổi: hộp `a` đặt được **vào trong (hoặc vừa khít)** hộp `b` nếu `a.w <= b.w` **và** `a.h <= b.h` (cho phép bằng). Hai hộp giống hệt nhau cũng lồng được. Tìm chuỗi dài nhất.

Câu hỏi suy nghĩ trước khi code: (1) còn cần sắp `h` giảm khi `w` bằng nhau không? (2) dùng `bisect_left` hay `bisect_right`?

In [35]:
def max_boxes_non_strict(boxes):
    # TODO: chọn key sắp xếp phù hợp với luật "<=" ở cả hai chiều
    # TODO: LIS KHÔNG GIẢM trên h (bisect nào?)
    raise NotImplementedError

# assert max_boxes_non_strict([[1, 1], [1, 1], [1, 1]]) == 3
# assert max_boxes_non_strict([[1, 1], [1, 2], [1, 3]]) == 3
# assert max_boxes_non_strict([[5, 4], [6, 4], [6, 7], [2, 3]]) == 4
# assert max_boxes_non_strict([[3, 2], [2, 3]]) == 1

### Bài 6 — Mini challenge: phong bì 3 chiều (không xoay)

Mỗi hộp là `[w, h, d]`. Hộp `a` lồng vào `b` nếu **cả ba** chiều của `a` nhỏ hơn hẳn `b`. Không được xoay. `n ≤ 300`. Tìm số hộp nhiều nhất lồng được.

Gợi ý suy nghĩ: mẹo `(w, -h)` chỉ "khử" được **một** chiều. Với 3 chiều, sau khi sắp theo `w`, hai chiều `h, d` còn lại không thể gộp vào một dãy số → quay về **DP `O(n²)`** của Cách 2 (so đủ cả ba chiều). Đây là ví dụ cho thấy vì sao biết cả cách `O(n²)` vẫn có ích.

In [36]:
def max_boxes_3d(boxes):
    # TODO: sắp theo w (hoặc theo cả bộ ba)
    # TODO: dp[i] = 1 + max(dp[j]) với j < i và box j lồng được vào box i ở CẢ 3 chiều
    raise NotImplementedError

# assert max_boxes_3d([[1, 1, 1], [2, 2, 2], [3, 3, 3]]) == 3
# assert max_boxes_3d([[1, 2, 3], [2, 3, 1], [3, 1, 2]]) == 1
# assert max_boxes_3d([[1, 1, 1], [2, 2, 1], [2, 3, 2], [3, 4, 3]]) == 3

## 22. Gợi ý theo 3 mức

**Bài 1**
- Mức 1: sắp theo `w` trước; nhóm cùng `w` thì `h` lớn đứng trước.
- Mức 2: nhóm `w = 1` có `[1,9], [1,5]`; nhóm `w = 3` có `[3,7], [3,4]`.
- Mức 3: dãy `h` = `[9, 5, 2, 7, 4]`; chạy `tails` trên dãy đó.

**Bài 2**
- Mức 1: `tails` luôn tăng hẳn; mỗi bước chỉ append hoặc thay **đúng một** ô.
- Mức 2: `x = 2` thay `5`; `x = 3` thay `6`.
- Mức 3: `tails` cuối là `[2, 3, 6, 7]`; so vị trí các số đó trong `H`.

**Bài 3**
- Mức 1: đúng 3 luật ở phần 12.
- Mức 2: `k = bisect_left(tails, x)`; so `k` với `len(tails)`.
- Mức 3: `return len(tails)` sau vòng lặp.

**Bài 4**
- Mức 1: hai bước — sort rồi LIS.
- Mức 2: `key=lambda e: (e[0], -e[1])`; trong vòng `for _, h in envs`.
- Mức 3: tái sử dụng đúng code Bài 3 cho dãy `h`.

**Bài 5**
- Mức 1: giờ hai hộp cùng `w` **được** nằm chung chuỗi (nếu `h` không giảm). Có còn cần "cấm hòa" không?
- Mức 2: sắp `(w tăng, h tăng)` để hộp cùng `w` xếp theo `h` không giảm; cần LIS **không giảm**.
- Mức 3: `sorted(boxes)` + `bisect_right`.

**Bài 6**
- Mức 1: sau khi sắp theo `w`, hộp lồng được vào `i` chỉ có thể đứng trước `i`.
- Mức 2: vẫn phải so **cả** `w` (vì `w` bằng nhau) cùng `h`, `d`.
- Mức 3: code giống `max_envelopes_dp` nhưng điều kiện có 3 vế `<`.

## 23. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
s      = [[1, 9], [1, 5], [2, 2], [3, 7], [3, 4]]
dãy h  = [9, 5, 2, 7, 4]
tails: 9 -> [9]; 5 -> [5]; 2 -> [2]; 7 -> [2, 7]; 4 -> [2, 4]
đáp án = 2   (ví dụ [2,2] -> [3,7])
```

### Đáp án Bài 2

| x | k | Hành động | `tails` sau |
|---:|---:|---|---|
| 5 | 0 | append | `[5]` |
| 2 | 0 | thay 5 | `[2]` |
| 8 | 1 | append | `[2, 8]` |
| 6 | 1 | thay 8 | `[2, 6]` |
| 3 | 1 | thay 6 | `[2, 3]` |
| 6 | 2 | append | `[2, 3, 6]` |
| 9 | 3 | append | `[2, 3, 6, 9]` |
| 7 | 3 | thay 9 | `[2, 3, 6, 7]` |

LIS = **4**. Ở đây `tails` cuối `[2, 3, 6, 7]` **tình cờ** là một dãy con thật của `H` (vị trí 1, 4, 5, 7) — nhưng đó chỉ là may mắn; ví dụ `[3, 4, 5, 1]` ở phần 12 cho thấy điều này không luôn đúng. Cell dưới kiểm chứng cả hai bài.

### Đáp án Bài 5 — vì sao `sorted(boxes)` + `bisect_right`?

- Với luật `<=`, hai hộp cùng `w` **được phép** đứng chung chuỗi nếu `h` không giảm. Vậy ta **không** cấm hòa nữa: sắp `(w tăng, h tăng)` để trong nhóm cùng `w` các `h` xếp không giảm, LIS được chọn bao nhiêu tùy ý.
- Hai phần tử liền nhau `i < j` trong dãy con không giảm của `h`: `w_i <= w_j` (vì đã sắp), `h_i <= h_j` → lồng được ✓. Ngược lại, mỗi chuỗi hợp lệ đều là dãy con của list đã sắp theo `(w, h)`.
- "Không giảm" → `bisect_right` (phần 19).

### Đáp án Bài 6 — DP `O(n²)` 3 chiều

Sắp theo bộ ba (w trước). Hộp `j` lồng được vào `i` thì `w_j < w_i` → `j` đứng trước `i`. DP như Cách 2 với điều kiện 3 vế. Kiểm chứng bằng brute force tập con trên input ngẫu nhiên nhỏ.

In [37]:
envs = [[3, 4], [1, 9], [3, 7], [2, 2], [1, 5]]
s = sorted(envs, key=lambda e: (e[0], -e[1]))
assert s == [[1, 9], [1, 5], [2, 2], [3, 7], [3, 4]]
assert [h for _, h in s] == [9, 5, 2, 7, 4]
assert max_envelopes_lis(envs) == 2 == max_envelopes_subsets(envs)

H = [5, 2, 8, 6, 3, 6, 9, 7]
tails = []
for x in H:
    k = bisect_left(tails, x)
    if k == len(tails):
        tails.append(x)
    else:
        tails[k] = x
assert tails == [2, 3, 6, 7] and len(tails) == lis_bruteforce(H) == 4
print("Đáp án Bài 1, Bài 2: PASS")

Đáp án Bài 1, Bài 2: PASS


In [38]:
# ---------- Bài 3 ----------
def answer_lis_length(nums):
    tails = []
    for x in nums:
        k = bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

assert answer_lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert answer_lis_length([0, 1, 0, 3, 2, 3]) == 4
assert answer_lis_length([7, 7, 7, 7]) == 1

# ---------- Bài 4 ----------
def answer_max_envelopes(envelopes):
    envs = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    return answer_lis_length([h for _, h in envs])

assert answer_max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
assert answer_max_envelopes([[1, 1], [1, 1], [1, 1]]) == 1
assert answer_max_envelopes([[1, 1], [1, 2], [1, 3]]) == 1
assert answer_max_envelopes([[1, 3], [2, 3]]) == 1
print("Đáp án Bài 3, Bài 4: PASS")


# ---------- Bài 5 ----------
def answer_max_boxes_non_strict(boxes):
    boxes = sorted(boxes)                     # (w tăng, h tăng)
    tails = []
    for _, h in boxes:
        k = bisect_right(tails, h)            # cho phép bằng
        if k == len(tails):
            tails.append(h)
        else:
            tails[k] = h
    return len(tails)

def brute_non_strict(boxes):
    best = 0
    s = sorted(boxes)
    for r in range(1, len(s) + 1):
        for g in combinations(s, r):
            if all(g[i][0] <= g[i + 1][0] and g[i][1] <= g[i + 1][1] for i in range(r - 1)):
                best = max(best, r)
    return best

assert answer_max_boxes_non_strict([[1, 1], [1, 1], [1, 1]]) == 3
assert answer_max_boxes_non_strict([[1, 1], [1, 2], [1, 3]]) == 3
assert answer_max_boxes_non_strict([[5, 4], [6, 4], [6, 7], [2, 3]]) == 4
assert answer_max_boxes_non_strict([[3, 2], [2, 3]]) == 1

# ---------- Bài 6 ----------
def answer_max_boxes_3d(boxes):
    boxes = sorted(boxes)
    n = len(boxes)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if all(boxes[j][d] < boxes[i][d] for d in range(3)):
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

def brute_3d(boxes):
    best = 0
    s = sorted(boxes)
    for r in range(1, len(s) + 1):
        for g in combinations(s, r):
            if all(all(g[i][d] < g[i + 1][d] for d in range(3)) for i in range(r - 1)):
                best = max(best, r)
    return best

assert answer_max_boxes_3d([[1, 1, 1], [2, 2, 2], [3, 3, 3]]) == 3
assert answer_max_boxes_3d([[1, 2, 3], [2, 3, 1], [3, 1, 2]]) == 1
assert answer_max_boxes_3d([[1, 1, 1], [2, 2, 1], [2, 3, 2], [3, 4, 3]]) == 3

random.seed(56)
for _ in range(300):
    n = random.randint(1, 8)
    b2 = [[random.randint(1, 4), random.randint(1, 4)] for _ in range(n)]
    assert answer_max_boxes_non_strict(b2) == brute_non_strict(b2)
    b3 = [[random.randint(1, 4) for _ in range(3)] for _ in range(n)]
    assert answer_max_boxes_3d(b3) == brute_3d(b3)
print("Đáp án Bài 5, Bài 6: PASS (kể cả 300 test ngẫu nhiên so với brute force)")

Đáp án Bài 3, Bài 4: PASS
Đáp án Bài 5, Bài 6: PASS (kể cả 300 test ngẫu nhiên so với brute force)


## 24. Template ngắn nên nhớ

```python
from bisect import bisect_left

class Solution:
    def maxEnvelopes(self, envelopes: List[List[int]]) -> int:
        envelopes.sort(key=lambda e: (e[0], -e[1]))   # w tăng, h GIẢM khi w bằng
        tails = []                                    # tails[k] = h cuối nhỏ nhất của chuỗi dài k+1
        for _, h in envelopes:
            k = bisect_left(tails, h)                 # tăng HẲN -> bisect_left
            if k == len(tails):
                tails.append(h)
            else:
                tails[k] = h
        return len(tails)
```

Công thức một dòng để nhớ: **"sort `(w, -h)` → LIS tăng hẳn trên `h` bằng `bisect_left` → `len(tails)`"**.

Khung tổng quát "sắp + LIS" cho các bài khác:

1. Xác định chiều nào sắp trước (chiều 1) và chiều nào chạy LIS (chiều 2).
2. Luật ngặt (`<`) → hòa ở chiều 1 thì sắp chiều 2 **ngược**, dùng `bisect_left`.
3. Luật không ngặt (`<=`) → hòa ở chiều 1 thì sắp chiều 2 **xuôi**, dùng `bisect_right`.
4. Từ 3 chiều trở lên → thường quay về DP `O(n²)`.

## 25. Checklist trước khi submit

- [ ] Key sắp xếp là `(e[0], -e[1])` — **có dấu trừ** ở `h`?
- [ ] Dùng `bisect_left` (tăng hẳn), **không** phải `bisect_right`?
- [ ] `tails` chỉ chứa **chiều cao `h`**, không phải cả cặp `[w, h]`?
- [ ] Trả về `len(tails)`, không phải `tails`?
- [ ] Không xoay phong bì (không đổi chỗ `w`, `h`)?
- [ ] Đã thử: Example 1 (3), Example 2 (1), cùng rộng `[[1,1],[1,2],[1,3]]` (1), cùng cao `[[1,3],[2,3]]` (1), `n = 1`?
- [ ] Độ phức tạp `O(n log n)` — không còn vòng lặp lồng `O(n²)` nào sót lại cho `n = 10^5`?